## SECTION 1

In [1]:
# Section 1.2 — Install Pinned Dependencies

%pip install -q --no-cache-dir \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "IndicTransToolkit==1.1.1" \
    peft accelerate sacrebleu sentencepiece kagglehub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.93.2 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Section 1.3 — Environment and GPU Verification

import sys
import importlib.metadata as metadata
import torch

print("=== Package Version Verification ===")

packages = [
    "transformers",
    "tokenizers",
    "IndicTransToolkit",
    "torch",
    "peft",
    "accelerate",
    "sacrebleu",
    "sentencepiece",
]

for package in packages:
    try:
        print(f"{package:22s}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:22s}: NOT INSTALLED")

print("\n=== Runtime Information ===")
print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory / (1024**3)
    )
    print(f"GPU memory: {gpu_memory:.2f} GiB")

assert metadata.version("transformers") == "4.46.3"
assert metadata.version("tokenizers") == "0.20.3"

print("\nEnvironment verification passed.")

=== Package Version Verification ===
transformers          : 4.46.3
tokenizers            : 0.20.3
IndicTransToolkit     : 1.1.1
torch                 : 2.11.0+cu128
peft                  : 0.20.0
accelerate            : 1.14.0
sacrebleu             : 2.6.0
sentencepiece         : 0.2.2

=== Runtime Information ===
Python version: 3.13.15
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GiB

Environment verification passed.


## Section 2 — Load the Shared Preprocessed Dataset


In [3]:

# Section 2.2 — Locate the Published Dataset (Bhili)

from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("Top-level input folders:")
for path in sorted(INPUT_ROOT.iterdir()):
    print(" -", path.name)

# Find the Bhili training split
matches = [
    path
    for path in INPUT_ROOT.rglob("bhili_train.parquet")
    if path.parent.name == "splits"
]

assert len(matches) == 1, (
    f"Expected one bhili_train.parquet file, found {len(matches)}. "
    "Check that your published preprocessing dataset is attached "
    "to this notebook."
)

SPLIT_DIR = matches[0].parent
DATASET_DIR = SPLIT_DIR.parent

print("\nDataset directory:", DATASET_DIR)
print("Split directory:", SPLIT_DIR)

print("\nAvailable split files:")
for path in sorted(SPLIT_DIR.iterdir()):
    print(" -", path.name)

# Verify that all three Bhili splits exist
for split_name in ["train", "validation", "test"]:
    split_path = SPLIT_DIR / f"bhili_{split_name}.parquet"
    assert split_path.exists(), f"Missing split: {split_path}"
    print(f"Verified: {split_path.name}")


Top-level input folders:
 - datasets

Dataset directory: /kaggle/input/datasets/ruchirjoshii/common-tribal-languages-preprocessing
Split directory: /kaggle/input/datasets/ruchirjoshii/common-tribal-languages-preprocessing/splits

Available split files:
 - bhili_test.csv
 - bhili_test.parquet
 - bhili_train.csv
 - bhili_train.parquet
 - bhili_validation.csv
 - bhili_validation.parquet
 - gondi_test.csv
 - gondi_test.parquet
 - gondi_train.csv
 - gondi_train.parquet
 - gondi_validation.csv
 - gondi_validation.parquet
 - mundari_test.csv
 - mundari_test.parquet
 - mundari_train.csv
 - mundari_train.parquet
 - mundari_validation.csv
 - mundari_validation.parquet
 - santali_test.csv
 - santali_test.parquet
 - santali_train.csv
 - santali_train.parquet
 - santali_validation.csv
 - santali_validation.parquet
Verified: bhili_train.parquet
Verified: bhili_validation.parquet
Verified: bhili_test.parquet


## Section 3 — Data Validation and Split Integrity


In [4]:

# Section 3.2 — Validate Bhili Data and Splits

import pandas as pd

language = "bhili"
source_col, target_col = ("Hindi", "Bhili")
split_names = ("train", "validation", "test")

# Load the three Bhili splits
all_data = {
    split: pd.read_parquet(
        SPLIT_DIR / f"{language}_{split}.parquet"
    )
    for split in split_names
}

validation_results = []

for split_name, df in all_data.items():
    checks = {
        "expected_columns": (
            source_col in df.columns
            and target_col in df.columns
        ),
        "missing_values": (
            not df[[source_col, target_col]].isna().any().any()
        ),
        "empty_sentences": (
            not df[source_col].astype(str).str.strip().eq("").any()
            and not df[target_col].astype(str).str.strip().eq("").any()
        ),
        "string_types": (
            df[source_col].map(lambda x: isinstance(x, str)).all()
            and df[target_col].map(lambda x: isinstance(x, str)).all()
        ),
        "duplicate_pairs": (
            not df.duplicated(
                subset=[source_col, target_col]
            ).any()
        ),
    }

    validation_results.append({
        "language": language,
        "split": split_name,
        "rows": len(df),
        **checks,
    })

# Check exact source-target pair overlap across splits
pair_sets = {
    split: set(
        df[[source_col, target_col]].itertuples(
            index=False, name=None
        )
    )
    for split, df in all_data.items()
}

overlap_counts = {
    "train_validation": len(pair_sets["train"] & pair_sets["validation"]),
    "train_test": len(pair_sets["train"] & pair_sets["test"]),
    "validation_test": len(pair_sets["validation"] & pair_sets["test"]),
}

print("BHILI — Exact pair overlap")
for comparison, count in overlap_counts.items():
    print(f"{comparison}: {count}")

assert all(count == 0 for count in overlap_counts.values()), (
    "Exact source-target pairs overlap across splits."
)

results_df = pd.DataFrame(validation_results)

print("\n=== Bhili Split Validation Results ===")
display(results_df)

check_columns = [
    "expected_columns",
    "missing_values",
    "empty_sentences",
    "string_types",
    "duplicate_pairs",
]

all_checks_passed = results_df[check_columns].all().all()

print("\nAll individual split checks passed:", all_checks_passed)

assert all_checks_passed, (
    "Validation failed. Inspect the results before proceeding."
)

print("Bhili data validation passed.")


BHILI — Exact pair overlap
train_validation: 0
train_test: 0
validation_test: 0

=== Bhili Split Validation Results ===


,language,split,rows,expected_columns,missing_values,empty_sentences,string_types,duplicate_pairs
0,bhili,train,17817,True,True,True,True,True
1,bhili,validation,990,True,True,True,True,True
2,bhili,test,990,True,True,True,True,True



All individual split checks passed: True
Bhili data validation passed.


## 4. Model Selection and Tokenization Research


In [5]:

# Section 4.1 — Environment and Existing Research Objects (Bhili)

import torch
import transformers
import tokenizers
import pandas as pd

print("Python/PyTorch environment")
print("---------------------------")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024**3),
            2
        ),
        "GiB"
    )

print("\nExisting research objects")
print("-------------------------")

for name in [
    "all_data",
    "source_col",
    "target_col",
    "SPLIT_DIR",
    "tokenizer",
    "processor",
    "MODEL_NAME",
]:
    print(f"{name}: {'available' if name in globals() else 'not defined'}")

if "all_data" in globals():
    print("\nBhili dataset split sizes:")
    print({split: len(df) for split, df in all_data.items()})

if "MODEL_NAME" in globals():
    print("\nPreviously selected model checkpoint:", MODEL_NAME)


Python/PyTorch environment
---------------------------
PyTorch: 2.11.0+cu128
Transformers: 4.46.3
Tokenizers: 0.20.3
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GiB

Existing research objects
-------------------------
all_data: available
source_col: available
target_col: available
SPLIT_DIR: available
tokenizer: not defined
processor: not defined
MODEL_NAME: not defined

Bhili dataset split sizes:
{'train': 17817, 'validation': 990, 'test': 990}


In [6]:

print("all_data type:", type(all_data))
print("all_data keys:", list(all_data.keys()))

print("\nAvailable variables related to Bhili:")
for name in ["SPLIT_DIR", "DATASET_DIR", "language", "splits", "all_data"]:
    if name in globals():
        value = globals()[name]
        print(f"{name}: {value if name not in ['all_data', 'splits'] else type(value)}")


all_data type: <class 'dict'>
all_data keys: ['train', 'validation', 'test']

Available variables related to Bhili:
SPLIT_DIR: /kaggle/input/datasets/ruchirjoshii/common-tribal-languages-preprocessing/splits
DATASET_DIR: /kaggle/input/datasets/ruchirjoshii/common-tribal-languages-preprocessing
language: bhili
all_data: <class 'dict'>


In [7]:

import pandas as pd

language = "bhili"
source_col, target_col = ("Hindi", "Bhili")
split_names = ("train", "validation", "test")

all_data = {
    language: {
        split: pd.read_parquet(
            SPLIT_DIR / f"{language}_{split}.parquet"
        )
        for split in split_names
    }
}

print("Loaded Bhili splits:")
for split, df in all_data[language].items():
    print(f"{split}: {len(df)} rows")


Loaded Bhili splits:
train: 17817 rows
validation: 990 rows
test: 990 rows


In [8]:

# Section 4.2 — Verify Bhili Dataset Columns and Translation Direction

expected_splits = {"train", "validation", "test"}

language = "bhili"
source_col, target_col = ("Hindi", "Bhili")
splits = all_data[language]

print("Dataset direction and schema verification")
print("=" * 55)

assert set(splits.keys()) == expected_splits, (
    f"Unexpected split names: {list(splits.keys())}"
)

print(f"\nLanguage: {language.upper()}")
print(f"Translation direction: {source_col} -> {target_col}")

for split_name in ("train", "validation", "test"):
    df = splits[split_name]

    missing_cols = {source_col, target_col} - set(df.columns)

    assert not missing_cols, (
        f"{language}/{split_name}: missing columns {missing_cols}"
    )

    assert df[[source_col, target_col]].notna().all().all(), (
        f"{language}/{split_name}: missing source or target text"
    )

    assert len(df) > 0, (
        f"{language}/{split_name}: split is empty"
    )

    print(
        f"  {split_name:10s}: {len(df):5d} rows | "
        f"source={source_col}, target={target_col}"
    )

# Display an example to verify the translation direction
example = splits["train"].iloc[0]

print("\nExample source (Hindi):", str(example[source_col])[:200])
print("Example target (Bhili):", str(example[target_col])[:200])

print("\nAll Bhili dataset direction and schema checks passed.")


Dataset direction and schema verification

Language: BHILI
Translation direction: Hindi -> Bhili
  train     : 17817 rows | source=Hindi, target=Bhili
  validation:   990 rows | source=Hindi, target=Bhili
  test      :   990 rows | source=Hindi, target=Bhili

Example source (Hindi): इंफ्रास्ट्रक्चर और टेक्नोलॉजी निवेश में वृद्धि हो रही है।
Example target (Bhili): इन्‍फ्रास्‍ट्रक्‍चर मानकों नी रीतें एआई अनुसंधान अनें विकास करवा नों प्रोत्‍साहन मळिरों से।

All Bhili dataset direction and schema checks passed.


In [9]:

# Section 4.3 — Restore Hugging Face Authentication

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

assert hf_token, (
    "HF_TOKEN is empty. Check that your Hugging Face token "
    "is saved in Kaggle Secrets under the name HF_TOKEN."
)

login(token=hf_token)

print("Hugging Face authentication restored.")


Hugging Face authentication restored.


In [10]:

# Section 4.4 — Inspect Candidate Checkpoint Configurations for Bhili

from transformers import AutoConfig

CANDIDATE_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "Indic -> Indic distilled": "ai4bharat/indictrans2-indic-indic-dist-320M",
    "English -> Indic 1B (comparison)": "ai4bharat/indictrans2-en-indic-1B",
    "Indic -> English 1B (comparison)": "ai4bharat/indictrans2-indic-en-1B",
    "English -> Indic distilled (comparison)": "ai4bharat/indictrans2-en-indic-dist-200M",
}

config_results = {}

for label, model_id in CANDIDATE_MODELS.items():
    print(f"\n{'=' * 70}\n{label}\n{model_id}")

    try:
        config = AutoConfig.from_pretrained(
            model_id,
            trust_remote_code=True,
            token=hf_token,
        )

        config_results[label] = config


        for field in (
            "model_type",
            "architectures",
            "is_encoder_decoder",
            "vocab_size",
            "tgt_vocab_size",
            "d_model",
            "encoder_layers",
            "decoder_layers",
        ):
            print(f"{field}: {getattr(config, field, None)}")

        for key in ("src_lang", "tgt_lang", "languages", "lang_code_to_id"):
            value = getattr(config, key, None)
            if value is not None:
                if isinstance(value, dict) and len(value) > 20:
                    value = f"<mapping with {len(value)} entries>"
                print(f"{key}: {value}")

    except Exception as e:
        print("CONFIG INSPECTION FAILED:", type(e).__name__, str(e)[:400])

print("\nConfiguration inspection complete.")



Indic -> Indic 1B
ai4bharat/indictrans2-indic-indic-1B


config.json: 0.00B [00:00, ?B/s]

configuration_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B:
- configuration_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model_type: IndicTrans
architectures: ['IndicTransForConditionalGeneration']
is_encoder_decoder: True
vocab_size: 122672
tgt_vocab_size: None
d_model: None
encoder_layers: 18
decoder_layers: 18

Indic -> Indic distilled
ai4bharat/indictrans2-indic-indic-dist-320M


...

model_type: IndicTrans
architectures: ['IndicTransForConditionalGeneration']
is_encoder_decoder: True
vocab_size: 32296
tgt_vocab_size: None
d_model: None
encoder_layers: 18
decoder_layers: 18

English -> Indic distilled (comparison)
ai4bharat/indictrans2-en-indic-dist-200M


config.json: 0.00B [00:00, ?B/s]

configuration_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- configuration_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model_type: IndicTrans
architectures: ['IndicTransForConditionalGeneration']
is_encoder_decoder: True
vocab_size: 122672
tgt_vocab_size: None
d_model: None
encoder_layers: 18
decoder_layers: 18

Configuration inspection complete.


In [11]:

# Section 4.3.2 — Compare Candidate Model Configurations

import pandas as pd
from transformers import AutoConfig

fields_to_compare = [
    "d_model",
    "hidden_size",
    "encoder_embed_dim",
    "decoder_embed_dim",
    "encoder_ffn_dim",
    "decoder_ffn_dim",
    "encoder_layers",
    "decoder_layers",
    "encoder_attention_heads",
    "decoder_attention_heads",
    "vocab_size",
    "tgt_vocab_size",
    "max_position_embeddings",
    "tie_word_embeddings",
]

comparison_rows = []

for label, model_id in CANDIDATE_MODELS.items():
    config = config_results.get(label)

    if config is None:
        try:
            config = AutoConfig.from_pretrained(
                model_id,
                trust_remote_code=True,
                token=hf_token,
            )
            config_results[label] = config
        except Exception as e:
            print(f"Could not inspect {label}: {type(e).__name__}: {e}")
            continue

    row = {
        "candidate": label,
        "model_id": model_id,
    }

    for field in fields_to_compare:
        value = getattr(config, field, None)

        if isinstance(value, (list, dict)):
            value = f"<{type(value).__name__}: {len(value)} items>"

        row[field] = value

    comparison_rows.append(row)

if comparison_rows:
    config_comparison = pd.DataFrame(comparison_rows).set_index("candidate")
    display(config_comparison.T)
else:
    print("No model configurations could be loaded.")


candidate,Indic -> Indic 1B,Indic -> Indic distilled,English -> Indic 1B (comparison),Indic -> English 1B (comparison),English -> Indic distilled (comparison)
model_id,ai4bharat/indictrans2-indic-indic-1B,ai4bharat/indictrans2-indic-indic-dist-320M,ai4bharat/indictrans2-en-indic-1B,ai4bharat/indictrans2-indic-en-1B,ai4bharat/indictrans2-en-indic-dist-200M
d_model,None,None,None,None,None
hidden_size,None,None,None,None,None
encoder_embed_dim,1024,512,1024,1024,512
decoder_embed_dim,1024,512,1024,1024,512
encoder_ffn_dim,8192,2048,8192,8192,2048
decoder_ffn_dim,8192,2048,8192,8192,2048
encoder_layers,18,18,18,18,18
decoder_layers,18,18,18,18,18
encoder_attention_heads,16,8,16,16,8


In [12]:

# Section 4.4 — Inspect Candidate Tokenizers for Bhili

from transformers import AutoTokenizer

tokenizer_results = {}

BHILI_SAMPLES = all_data["bhili"]["train"]["Bhili"].head(3).tolist()

for label, model_id in CANDIDATE_MODELS.items():
    print(f"\n{'=' * 70}")
    print(label)
    print(model_id)

    try:
        tok = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=True,
            token=hf_token,
        )
        tokenizer_results[label] = tok

        print("Tokenizer class:", type(tok).__name__)
        print("len(tokenizer):", len(tok))
        print("Tokenizer vocab_size:", getattr(tok, "vocab_size", None))
        print("Special tokens:", tok.special_tokens_map)

        for attr in ("src_spm", "tgt_spm"):
            spm = getattr(tok, attr, None)
            if spm is not None:
                try:
                    print(f"{attr} piece count:", spm.GetPieceSize())
                except Exception:
                    print(f"{attr}: available, piece count unavailable")
            else:
                print(f"{attr}: not exposed")

        print("Special token IDs:")
        for token_name in (
            "bos_token_id",
            "eos_token_id",
            "pad_token_id",
            "unk_token_id",
        ):
            print(f"  {token_name}:", getattr(tok, token_name, None))

        for attr in (
            "lang_code_to_id",
            "supported_languages",
            "src_lang",
            "tgt_lang",
        ):
            value = getattr(tok, attr, None)
            if value is not None:
                if isinstance(value, dict):
                    print(f"{attr}: {len(value)} entries")
                    print("  Sample:", list(value.items())[:8])
                else:
                    print(f"{attr}:", value)

        # Generic tokenizer probe. Use verified Bhili examples in the
        # actual experiment; these strings are only diagnostic placeholders.
        print("Tokenizer probe:")
        for sample in BHILI_SAMPLES:
            encoded = tok(
                sample,
                add_special_tokens=False,
                truncation=False,
            )
            pieces = tok.convert_ids_to_tokens(encoded["input_ids"])
            print("  Text:", sample)
            print("  Token IDs:", encoded["input_ids"])
            print("  Pieces:", pieces)
            print("  Token count:", len(encoded["input_ids"]))

    except Exception as e:
        print("TOKENIZER INSPECTION FAILED:", type(e).__name__, str(e)[:350])



Indic -> Indic 1B
ai4bharat/indictrans2-indic-indic-1B


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

...

dict.TGT.json: 0.00B [00:00, ?B/s]

model.SRC:   0%|          | 0.00/759k [00:00<?, ?B/s]

model.TGT:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

Tokenizer class: IndicTransTokenizer
len(tokenizer): 32322
Tokenizer vocab_size: 32322
Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}
src_spm piece count: 32000
tgt_spm piece count: 128000
Special token IDs:
  bos_token_id: 0
  eos_token_id: 2
  pad_token_id: 1
  unk_token_id: 3
Tokenizer probe:
TOKENIZER INSPECTION FAILED: AssertionError Invalid source language tag: इन्‍फ्रास्‍ट्रक्‍चर


In [13]:

# Section 4.5 — Compare SentencePiece Fertility for Bhili

import numpy as np
import pandas as pd

SAMPLE_SIZE = 2000
RANDOM_SEED = 42

# Bhili target text from the training split
bhili_df = all_data["bhili"]["train"]
bhili_texts = (
    bhili_df["Bhili"]
    .dropna()
    .astype(str)
)

bhili_texts = bhili_texts[
    bhili_texts.str.strip().ne("")
]

bhili_texts = bhili_texts.sample(
    n=min(SAMPLE_SIZE, len(bhili_texts)),
    random_state=RANDOM_SEED,
).tolist()

fertility_rows = []

for model_label in [
    "Indic -> Indic 1B",
    "Indic -> Indic distilled",
    "Indic -> English 1B (comparison)",
    "English -> Indic 1B (comparison)",
    "English -> Indic distilled (comparison)",
]:
    tok = tokenizer_results.get(model_label)

    if tok is None:
        print(f"Skipping {model_label}: tokenizer unavailable")
        continue

    # Compare source and target SentencePiece models on the same Bhili text.
    for spm_name in ["src_spm", "tgt_spm"]:
        spm = getattr(tok, spm_name, None)

        if spm is None:
            continue

        sentence_token_counts = []
        total_words = 0
        total_word_tokens = 0
        split_words = 0
        unknown_pieces = 0
        total_pieces = 0

        for text in bhili_texts:
            pieces = spm.EncodeAsPieces(text)
            sentence_token_counts.append(len(pieces))
            total_pieces += len(pieces)

            words = text.split()
            total_words += len(words)

            for word in words:
                word_pieces = spm.EncodeAsPieces(word)
                total_word_tokens += len(word_pieces)

                if len(word_pieces) > 1:
                    split_words += 1

                unknown_pieces += sum(
                    piece == "<unk>" for piece in word_pieces
                )

        fertility_rows.append({
            "language": "bhili",
            "text_column": "Bhili",
            "model": model_label,
            "spm": spm_name,
            "sentences": len(bhili_texts),
            "avg_tokens_per_sentence": round(
                np.mean(sentence_token_counts), 3
            ) if sentence_token_counts else 0,
            "tokens_per_word": round(
                total_word_tokens / max(total_words, 1), 3
            ),
            "word_split_rate_pct": round(
                100 * split_words / max(total_words, 1), 3
            ),
            "unknown_piece_rate_pct": round(
                100 * unknown_pieces / max(total_pieces, 1), 4
            ),
        })

fertility_df = pd.DataFrame(fertility_rows)

if not fertility_df.empty:
    display(
        fertility_df.sort_values(
            ["model", "spm"]
        ).reset_index(drop=True)
    )
else:
    print("No fertility results generated. Check tokenizer availability.")


,language,text_column,model,spm,sentences,avg_tokens_per_sentence,tokens_per_word,word_split_rate_pct,unknown_piece_rate_pct
0,bhili,Bhili,English -> Indic 1B (comparison),src_spm,2000,106.305,4.980,98.360,0.0
1,bhili,Bhili,English -> Indic 1B (comparison),tgt_spm,2000,30.902,1.448,31.921,0.0
2,bhili,Bhili,English -> Indic distilled (comparison),src_spm,2000,106.305,4.980,98.360,0.0
3,bhili,Bhili,English -> Indic distilled (comparison),tgt_spm,2000,30.902,1.448,31.921,0.0
4,bhili,Bhili,Indic -> English 1B (comparison),src_spm,2000,30.902,1.448,31.921,0.0
5,bhili,Bhili,Indic -> English 1B (comparison),tgt_spm,2000,106.305,4.980,98.360,0.0
6,bhili,Bhili,Indic -> Indic 1B,src_spm,2000,30.902,1.448,31.921,0.0
7,bhili,Bhili,Indic -> Indic 1B,tgt_spm,2000,30.902,1.448,31.921,0.0
8,bhili,Bhili,Indic -> Indic distilled,src_spm,2000,30.902,1.448,31.921,0.0
9,bhili,Bhili,Indic -> Indic distilled,tgt_spm,2000,30.902,1.448,31.921,0.0


In [14]:

# Section 4.5.1 — Fingerprint Actual SentencePiece Vocabularies

import hashlib
import pandas as pd

spm_fingerprints = {}

for model_name, tok in tokenizer_results.items():
    for spm_name in ("src_spm", "tgt_spm"):
        spm = getattr(tok, spm_name, None)

        if spm is None:
            print(f"Skipping {model_name}/{spm_name}: unavailable")
            continue

        piece_count = spm.GetPieceSize()
        hasher = hashlib.sha256()

        for piece_id in range(piece_count):
            piece = spm.IdToPiece(piece_id)
            hasher.update(piece.encode("utf-8"))
            hasher.update(b"\n")

        fingerprint = hasher.hexdigest()

        spm_fingerprints[(model_name, spm_name)] = {
            "model": model_name,
            "spm": spm_name,
            "piece_count": piece_count,
            "fingerprint": fingerprint,
        }

fingerprint_df = pd.DataFrame(spm_fingerprints.values())

if fingerprint_df.empty:
    print("No SentencePiece vocabularies were available.")
else:
    display(
        fingerprint_df[
            ["model", "spm", "piece_count", "fingerprint"]
        ].sort_values(
            ["piece_count", "model", "spm"]
        ).reset_index(drop=True)
    )

    print(
        "\nNumber of distinct SentencePiece vocabularies:",
        fingerprint_df["fingerprint"].nunique()
    )

    print("\nVocabulary groups:")
    display(
        fingerprint_df.groupby(
            ["piece_count", "fingerprint"], as_index=False
        ).agg(
            models=("model", lambda x: ", ".join(x)),
            tokenizer_sides=("spm", lambda x: ", ".join(x)),
            occurrences=("model", "size"),
        ).sort_values(
            ["piece_count", "occurrences"],
            ascending=[True, False]
        )
    )


,model,spm,piece_count,fingerprint
0,English -> Indic 1B (comparison),src_spm,32000,e72be62ef1ac73a3be7ba13f77d0e3f4c7e8c401d609b6...
1,English -> Indic distilled (comparison),src_spm,32000,e72be62ef1ac73a3be7ba13f77d0e3f4c7e8c401d609b6...
2,Indic -> English 1B (comparison),tgt_spm,32000,e72be62ef1ac73a3be7ba13f77d0e3f4c7e8c401d609b6...
3,English -> Indic 1B (comparison),tgt_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
4,English -> Indic distilled (comparison),tgt_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
5,Indic -> English 1B (comparison),src_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
6,Indic -> Indic 1B,src_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
7,Indic -> Indic 1B,tgt_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
8,Indic -> Indic distilled,src_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...
9,Indic -> Indic distilled,tgt_spm,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...



Number of distinct SentencePiece vocabularies: 2

Vocabulary groups:


,piece_count,fingerprint,models,tokenizer_sides,occurrences
0,32000,e72be62ef1ac73a3be7ba13f77d0e3f4c7e8c401d609b6...,"English -> Indic 1B (comparison), Indic -> Eng...","src_spm, tgt_spm, src_spm",3
1,128000,a3e3d0cd0747fcda72627bb8f535a8040f313f78eb36f6...,"Indic -> Indic 1B, Indic -> Indic 1B, Indic ->...","src_spm, tgt_spm, src_spm, tgt_spm, tgt_spm, s...",7


In [15]:

# Section 4.5.2 — Compare the Two Distinct Vocabularies (Bhili)

import pandas as pd
import numpy as np

# Select one representative tokenizer for each actual fingerprint.
unique_fingerprints = fingerprint_df.drop_duplicates(
    subset=["fingerprint"]
)

spm_32k = None
spm_128k = None

for _, row in unique_fingerprints.iterrows():
    tok = tokenizer_results[row["model"]]
    spm = getattr(tok, row["spm"])
    piece_count = spm.GetPieceSize()

    if piece_count == 32000:
        spm_32k = spm
    elif piece_count == 128000:
        spm_128k = spm

assert spm_32k is not None, "Could not locate the 32k vocabulary."
assert spm_128k is not None, "Could not locate the 128k vocabulary."

rows = []

language = "bhili"
target_col = "Bhili"

texts = (
    all_data[language]["train"][target_col]
    .dropna()
    .astype(str)
    .head(2000)
    .tolist()
)

for vocab_size, spm in [
    (32000, spm_32k),
    (128000, spm_128k),
]:
    total_tokens = 0
    total_words = 0
    split_words = 0

    for text in texts:
        total_tokens += len(spm.EncodeAsPieces(text))

        for word in text.split():
            total_words += 1
            if len(spm.EncodeAsPieces(word)) > 1:
                split_words += 1

    rows.append({
        "language": language,
        "vocabulary_size": vocab_size,
        "sentences": len(texts),
        "avg_tokens_per_sentence": round(
            total_tokens / max(len(texts), 1), 3
        ),
        "tokens_per_word": round(
            total_tokens / max(total_words, 1), 3
        ),
        "word_split_rate_pct": round(
            100 * split_words / max(total_words, 1), 3
        ),
    })

comparison_df = pd.DataFrame(rows)

display(
    comparison_df.sort_values(
        ["language", "vocabulary_size"]
    ).reset_index(drop=True)
)


,language,vocabulary_size,sentences,avg_tokens_per_sentence,tokens_per_word,word_split_rate_pct
0,bhili,32000,2000,107.505,4.981,98.281
1,bhili,128000,2000,31.233,1.447,32.111


In [16]:

# Section 4.6 — Check tokenizer objects and initialize IndicProcessor

from IndicTransToolkit import IndicProcessor

print("Candidate tokenizers available:", len(tokenizer_results))
print("Candidate checkpoints:", list(tokenizer_results.keys()))

print("\nLanguage-related tokenizer attributes:")
for label, tok in tokenizer_results.items():
    print(f"\n{label}")

    for attr in dir(tok):
        if any(term in attr.lower() for term in ("lang_code", "supported_languages")):
            try:
                value = getattr(tok, attr)
                if not callable(value):
                    print(f"  {attr}: {str(value)[:300]}")
            except Exception:
                pass

# Initialize the processor required by the next cell
processor = IndicProcessor(inference=True)

print("\nProcessor initialized:", type(processor).__name__)


Candidate tokenizers available: 5
Candidate checkpoints: ['Indic -> Indic 1B', 'Indic -> Indic distilled', 'English -> Indic 1B (comparison)', 'Indic -> English 1B (comparison)', 'English -> Indic distilled (comparison)']

Language-related tokenizer attributes:

Indic -> Indic 1B

Indic -> Indic distilled

English -> Indic 1B (comparison)

Indic -> English 1B (comparison)

English -> Indic distilled (comparison)

Processor initialized: IndicProcessor


In [17]:
# Section 4.6 — Inspect the compiled IndicProcessor API

import inspect

print("Processor methods and signatures:\n")

for name in dir(processor):
    if name.startswith("_"):
        continue

    try:
        obj = getattr(processor, name)
        if callable(obj):
            try:
                signature = inspect.signature(obj)
            except (TypeError, ValueError):
                signature = "(signature unavailable)"

            print(f"{name}{signature}")
    except Exception as e:
        print(f"{name}: inspection failed ({type(e).__name__})")

Processor methods and signatures:

postprocess_batch(sents, lang='hin_Deva', visualize=False, num_return_sequences=1)
preprocess_batch(batch, src_lang, tgt_lang=None, is_target=False, visualize=False)


In [18]:

# Section 4.6 — Search installed toolkit for language-code definitions

from pathlib import Path
import IndicTransToolkit

package_dir = Path(IndicTransToolkit.__file__).parent

print("Toolkit directory:", package_dir)

keywords = (
    "bhili", "gondi", "mundari", "santali",
    "hin_deva", "eng_latn", "language_codes"
)

matches = []

for path in package_dir.rglob("*"):
    if path.is_file() and path.suffix.lower() in {
        ".py", ".json", ".txt", ".yaml", ".yml"
    }:
        try:
            content = path.read_text(
                encoding="utf-8", errors="ignore"
            )
            found = [
                word for word in keywords
                if word in content.lower()
            ]
            if found:
                matches.append(
                    (str(path.relative_to(package_dir)), found)
                )
        except OSError:
            pass

if matches:
    for filename, found in matches:
        print(f"{filename}: {', '.join(sorted(set(found)))}")
else:
    print("No matching text files found in the installed toolkit.")


Toolkit directory: /usr/local/lib/python3.13/dist-packages/IndicTransToolkit
evaluator.py: eng_latn, hin_deva


In [19]:

# Section 4.6 — Inspect checkpoint language metadata

for name, config in config_results.items():
    print(f"\n{'=' * 60}")
    print(name)

    if isinstance(config, dict):
        for key in (
            "model_name", "name_or_path", "architectures",
            "model_type", "language_codes", "langs",
            "lang_code_to_id", "id_to_lang_code"
        ):
            if key in config:
                print(f"{key}: {config[key]}")
    else:
        print("Configuration object:", type(config).__name__)

        for key in (
            "name_or_path", "architectures", "model_type",
            "language_codes", "langs",
            "lang_code_to_id", "id_to_lang_code"
        ):
            value = getattr(config, key, None)
            if value is not None:
                print(f"{key}: {value}")



Indic -> Indic 1B
Configuration object: IndicTransConfig
name_or_path: ai4bharat/indictrans2-indic-indic-1B
architectures: ['IndicTransForConditionalGeneration']
model_type: IndicTrans

Indic -> Indic distilled
Configuration object: IndicTransConfig
name_or_path: ai4bharat/indictrans2-indic-indic-dist-320M
architectures: ['IndicTransForConditionalGeneration']
model_type: IndicTrans

English -> Indic 1B (comparison)
Configuration object: IndicTransConfig
name_or_path: ai4bharat/indictrans2-en-indic-1B
architectures: ['IndicTransForConditionalGeneration']
model_type: IndicTrans

Indic -> English 1B (comparison)
Configuration object: IndicTransConfig
name_or_path: ai4bharat/indictrans2-indic-en-1B
architectures: ['IndicTransForConditionalGeneration']
model_type: IndicTrans

English -> Indic distilled (comparison)
Configuration object: IndicTransConfig
name_or_path: ai4bharat/indictrans2-en-indic-dist-200M
architectures: ['IndicTransForConditionalGeneration']
model_type: IndicTrans


In [20]:
# Section 4.6 — Inspect tokenizer API for language handling

for name, tok in tokenizer_results.items():
    print(f"\n{'=' * 60}")
    print(name)
    print("Class:", type(tok).__name__)

    for attr in dir(tok):
        if any(term in attr.lower() for term in (
            "lang", "language", "src_spm", "tgt_spm",
            "preprocess", "special"
        )):
            try:
                value = getattr(tok, attr)
                if not callable(value):
                    print(f"{attr}: {str(value)[:200]}")
            except Exception:
                pass


Indic -> Indic 1B
Class: IndicTransTokenizer
SPECIAL_TOKENS_ATTRIBUTES: ['bos_token', 'eos_token', 'unk_token', 'sep_token', 'pad_token', 'cls_token', 'mask_token', 'additional_special_tokens']
_additional_special_tokens: []
additional_special_tokens: []
additional_special_tokens_ids: []
all_special_ids: [0, 2, 3, 1]
all_special_tokens: ['<s>', '</s>', '<unk>', '<pad>']
all_special_tokens_extended: [AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), Ad
special_tokens_map: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}
special_tokens_map_extended: {'bos_token': AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 'eos_token': AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalize
split_special_tokens: False
src_spm: <sentencepiece.Sentence

In [21]:

# Section 4.6 — Prepare one example for the Bhili translation task
# This cell only inspects the existing data; it does not train or load models.

language = "bhili"
src_col, tgt_col = "Hindi", "Bhili"

example = all_data[language]["validation"].iloc[0]

print(f"\n{'=' * 55}")
print(f"Task: {src_col} -> {tgt_col}")
print("Language key:", language)
print("Source column:", src_col)
print("Target column:", tgt_col)
print("Source example:", str(example[src_col])[:250])
print("Target example:", str(example[tgt_col])[:250])



Task: Hindi -> Bhili
Language key: bhili
Source column: Hindi
Target column: Bhili
Source example: दूसरे शब्दों में, समानता प्रत्येक व्यक्ति के माध्यम से प्राप्त की जाती है जो अपनी क्षमता के अनुसार खर्चों का भुगतान करता है।
Target example: बिजा ब्रोह शब्‍दों मा समानता एक व्‍यक्ति ना वचमाना से की आपडी ताकत अनुसार खर्चा करे।



# Section 4.6 Results and Analysis — Language and Model Compatibility Analysis

## 1. Research Objective

Our research investigates whether vocabulary extension and informed embedding initialization can improve neural machine translation for low-resource Indian tribal languages. This notebook focuses on **Bhili**, using Hindi as the source language and Bhili as the target language.

We compare two approaches:

1. **Baseline:** Fine-tune a pretrained IndicTrans2 model using LoRA while retaining its original vocabulary.
2. **Proposed method:** Fine-tune the same model using LoRA after extending its vocabulary and applying a selected embedding-initialization strategy.

For a fair comparison, both approaches must use the same pretrained checkpoint, training data, data splits, evaluation procedure, and comparable training settings. The main experimental question is whether vocabulary extension and embedding initialization improve the model's ability to generate Bhili text.

## 2. Translation Task and Dataset Example

The translation task investigated in this notebook is:

| Language | Translation direction | Source language | Target language |
|---|---|---|---|
| Bhili | Hindi → Bhili | Hindi | Bhili |

The validation dataset was inspected to confirm that the expected `Hindi` and `Bhili` columns are available. The example-printing cell displays a Hindi source sentence alongside its corresponding Bhili target sentence.

The Bhili text in the dataset uses its existing script representation, including Devanagari-script text. This representation is important when examining tokenizer behavior and deciding how to initialize embeddings for newly added vocabulary items.

Inspecting a validation example confirms the structure of the dataset, but does not establish translation quality or whether a pretrained checkpoint officially supports Bhili.

## 3. Findings from the Language-Support Investigation

The installed `IndicTransToolkit` package was searched for references to Bhili, Gondi, Mundari, Santali, and common language codes. The search returned a match in `evaluator.py` for `eng_latn` and `hin_deva`, but did not find explicit references to Bhili in the searched files.

The tokenizer inspection also examined language-related attributes, source and target SentencePiece objects, preprocessing attributes, and special tokens. No explicit Bhili-specific support was established through these checks.

The checkpoint configuration inspection was intended to identify language-related metadata, including language codes and language-to-ID mappings. These checks do not, by themselves, provide a definitive list of languages supported by every checkpoint.

The findings therefore lead to the following distinction:

- **Bhili:** Explicit support was not established by the toolkit and tokenizer inspections performed so far. Further verification of the selected checkpoint's language-tag handling and inference pipeline is necessary.
- **Tokenizer representation:** The inspected SentencePiece vocabulary can encode the Bhili samples without producing unknown pieces in the earlier fertility tests. This indicates that the text can be represented by the tokenizer; it does not establish that the model understands or can reliably translate Bhili.

A language's representation in a tokenizer, acceptance of a language tag, and actual translation capability are separate properties. Assigning an arbitrary language code would not, on its own, make an unsupported target language work.

## 4. Candidate Checkpoints

Five IndicTrans2 checkpoints were inspected as part of the initial model-comparison workflow.

| Checkpoint | Approximate model size | Intended translation direction |
|---|---:|---|
| `ai4bharat/indictrans2-indic-en-1B` | 1B | Indic → English |
| `ai4bharat/indictrans2-en-indic-1B` | 1B | English → Indic |
| `ai4bharat/indictrans2-indic-indic-1B` | 1B | Indic → Indic |
| `ai4bharat/indictrans2-en-indic-dist-200M` | 200M | English → Indic |
| `ai4bharat/indictrans2-indic-indic-dist-320M` | 320M | Indic → Indic |

The inspected checkpoints use an encoder-decoder architecture associated with `IndicTransForConditionalGeneration`.

The 1B checkpoints use 1024-dimensional encoder and decoder embeddings, 8192-dimensional feed-forward layers, 18 encoder layers, 18 decoder layers, and 16 attention heads.

The distilled checkpoints use 512-dimensional embeddings, 2048-dimensional feed-forward layers, 18 encoder layers, 18 decoder layers, and 8 attention heads.

These architectural differences make model capacity and computational cost relevant considerations. However, a larger checkpoint is not automatically better suited to Bhili, particularly if the required target-language handling is unsupported.

For the Hindi → Bhili task, the Indic → Indic checkpoints are the most directly relevant candidates by their intended translation direction. Their suitability must still be verified before selecting the final training checkpoint.

## 5. SentencePiece Vocabulary Comparison

The tokenizer-fingerprinting experiment identified **two distinct SentencePiece vocabularies** across the five candidate checkpoints:

- **32,000-piece vocabulary:** Used on English-facing sides of the inspected checkpoints.
- **128,000-piece vocabulary:** Used on Indic-facing sides of the inspected checkpoints.

The Indic → Indic checkpoints use the Indic-side vocabulary for both source and target processing. The English → Indic checkpoints use the English-side vocabulary for source processing and the Indic-side vocabulary for target processing. The Indic → English checkpoint uses the reverse arrangement.

The fingerprinting results indicate that the five checkpoints share these two underlying SentencePiece vocabulary types, although they have different model weights and, in some cases, different model capacities.

This distinction matters for the proposed vocabulary-extension experiment. The vocabulary used by the selected checkpoint determines the existing token inventory and the embedding rows that would need to be preserved when extending the vocabulary.

## 6. Bhili Tokenizer Fertility Results

The earlier fertility experiment compared how the Bhili target text was segmented by the source-side and target-side SentencePiece models of the candidate checkpoints.

For the Indic → Indic checkpoints, the Bhili samples produced the following results:

| Metric | Indic-side vocabulary |
|---|---:|
| Average tokens per sentence | 30.902 |
| Tokens per whitespace-separated word | 1.448 |
| Word split rate | 31.921% |
| Unknown-piece rate | 0% |

For the English → Indic checkpoints, the English-side vocabulary produced substantially more fragmentation on the same Bhili samples:

| Metric | English-side vocabulary |
|---|---:|
| Average tokens per sentence | 106.305 |
| Tokens per whitespace-separated word | 4.980 |
| Word split rate | 98.360% |
| Unknown-piece rate | 0% |

These measurements indicate that the Indic-side SentencePiece vocabulary represents the inspected Bhili text more compactly than the English-side vocabulary.

The zero unknown-piece rate means that the tested text could be segmented without producing unknown pieces under the tested vocabulary. It does not mean that the model has learned Bhili grammar, semantics, or translation behavior.

Lower fragmentation is a useful diagnostic for vocabulary-extension research, but it cannot be used as a substitute for evaluating translation quality.

## 7. Implications for Vocabulary Extension and Embedding Initialization

The diagnostics provide an initial basis for designing the Bhili experiments.

First, the Indic-side vocabulary is a more appropriate starting point than the English-side vocabulary for the inspected Bhili text because it produces substantially less fragmentation.

Second, the vocabulary-extension method must preserve the original vocabulary IDs and their associated pretrained embedding rows. New Bhili-specific tokens, if added, must receive valid new IDs and compatible embedding initialization.

Third, the tokenizer, model embedding matrix, output projection, and vocabulary configuration must remain consistent after extension. Any required changes must be applied consistently to the baseline and proposed experimental pipelines, except for the vocabulary-extension and initialization treatment being evaluated.

Finally, language-code and inference compatibility must be established before training. Vocabulary extension alone does not automatically teach a pretrained model to generate Bhili, and LoRA fine-tuning cannot be assumed to solve an unsupported language-tag or preprocessing problem.

## 8. Preliminary Conclusion

The Section 4.6 diagnostics establish that the Bhili dataset has the expected Hindi and Bhili columns, that the candidate tokenizers can represent the inspected Bhili samples, and that the Indic-side SentencePiece vocabulary segments those samples more compactly than the English-side vocabulary.

Among the five inspected checkpoints, the Indic → Indic models are the most relevant initial candidates for the Hindi → Bhili direction. However, explicit Bhili support has not yet been established by the inspections performed so far.

The next step is to verify the selected checkpoint's language-tag requirements and inference pipeline, then establish a working baseline before implementing vocabulary extension and embedding initialization. The final comparison should evaluate both approaches on the same held-out Bhili test set using the same evaluation procedure.

These findings are preliminary compatibility and tokenizer diagnostics, not evidence that either training approach already improves Bhili translation quality.


In [22]:

# Section 4.6 — Bhili tokenizer round-trip check
# Reuses the tokenizer already loaded in tokenizer_results.

tok = tokenizer_results["Indic -> Indic 1B"]

row = all_data["bhili"]["validation"].iloc[0]
source_text = row["Hindi"]
target_text = row["Bhili"]

# Encode the source using the Indic source SentencePiece model
source_pieces = tok.src_spm.encode(source_text, out_type=str)

# Encode the Bhili target using the Indic target SentencePiece model
target_pieces = tok.tgt_spm.encode(target_text, out_type=str)

# Decode each sequence back to text
source_roundtrip = tok.src_spm.decode(source_pieces)
target_roundtrip = tok.tgt_spm.decode(target_pieces)

print("Source SentencePiece pieces:", len(source_pieces))
print("Target SentencePiece pieces:", len(target_pieces))

print("\nOriginal Bhili target:")
print(target_text)

print("\nDecoded Bhili target:")
print(target_roundtrip)

print("\nBhili round-trip exact match:", target_text == target_roundtrip)
print("Source round-trip exact match:", source_text == source_roundtrip)


Source SentencePiece pieces: 26
Target SentencePiece pieces: 25

Original Bhili target:
बिजा ब्रोह शब्‍दों मा समानता एक व्‍यक्ति ना वचमाना से की आपडी ताकत अनुसार खर्चा करे।

Decoded Bhili target:
बिजा ब्रोह शब्‍दों मा समानता एक व्‍यक्ति ना वचमाना से की आपडी ताकत अनुसार खर्चा करे।

Bhili round-trip exact match: True
Source round-trip exact match: True


In [23]:

# Section 4.6 — Round-trip and tokenization check for Bhili

tokenizer_name = "Indic -> Indic 1B"
tok = tokenizer_results[tokenizer_name]

target_col = "Bhili"
df = all_data["bhili"]["validation"].head(100)

token_counts = []
exact_matches = 0

for text in df[target_col].astype(str):
    pieces = tok.tgt_spm.encode(text, out_type=str)
    decoded = tok.tgt_spm.decode(pieces)

    token_counts.append(len(pieces))
    exact_matches += (decoded == text)

print(f"\n{'=' * 55}")
print("Language: bhili")
print(f"Tokenizer: {tokenizer_name}")
print(f"Examples checked: {len(df)}")
print(
    f"Mean target SentencePiece tokens: "
    f"{sum(token_counts) / len(token_counts):.2f}"
    if token_counts else "Mean target SentencePiece tokens: N/A"
)
print(f"Exact round trips: {exact_matches}/{len(df)}")



Language: bhili
Tokenizer: Indic -> Indic 1B
Examples checked: 100
Mean target SentencePiece tokens: 32.60
Exact round trips: 93/100


In [24]:

# Section 4.6 — Inspect Bhili tokenizer round-trip failures

tokenizer_name = "Indic -> Indic 1B"
tok = tokenizer_results[tokenizer_name]
target_col = "Bhili"
df = all_data["bhili"]["validation"].head(100)

failures = []

for text in df[target_col].astype(str):
    pieces = tok.tgt_spm.encode(text, out_type=str)
    decoded = tok.tgt_spm.decode(pieces)

    if decoded != text:
        failures.append({
            "original": text,
            "decoded": decoded,
            "original_repr": repr(text),
            "decoded_repr": repr(decoded),
            "original_length": len(text),
            "decoded_length": len(decoded),
            "pieces": pieces,
        })

print(f"\n{'=' * 70}")
print(f"Language: bhili | Failures: {len(failures)}/{len(df)}")

for i, item in enumerate(failures[:3], start=1):
    print(f"\nFailure {i}")
    print("Original:     ", item["original"][:250])
    print("Decoded:      ", item["decoded"][:250])
    print("Original repr:", item["original_repr"][:300])
    print("Decoded repr: ", item["decoded_repr"][:300])
    print(
        "Lengths:      ",
        item["original_length"],
        "->",
        item["decoded_length"],
    )
    print("Pieces:       ", item["pieces"][:40])



Language: bhili | Failures: 7/100

Failure 1
Original:      हमु ने इना मुदा उपर हाते मलीने काम करवा हारू शंघाई सहयोग पार्टी पण एक खास ढ़ाचा आलवानों से ।
Decoded:       हमु ने इना मुदा उपर हाते मलीने काम करवा हारू शंघाई सहयोग पार्टी पण एक खास ढ़ाचा आलवानों से ।
Original repr: 'हमु ने इना मुदा उपर हाते मलीने काम करवा हारू शंघाई सहयोग पार्टी पण एक खास ढ़ाचा आलवानों से ।'
Decoded repr:  'हमु ने इना मुदा उपर हाते मलीने काम करवा हारू शंघाई सहयोग पार्टी पण एक खास ढ़ाचा आलवानों से ।'
Lengths:       91 -> 92
Pieces:        ['▁हम', 'ु', '▁ने', '▁इना', '▁मुदा', '▁उपर', '▁हाते', '▁म', 'ली', 'ने', '▁काम', '▁करवा', '▁हार', 'ू', '▁श', 'ंघ', 'ाई', '▁सहयोग', '▁पार्टी', '▁पण', '▁एक', '▁खास', '▁', 'ढ़ा', 'चा', '▁आल', 'वान', 'ों', '▁से', '▁।']

Failure 2
Original:      खोकडी ने तरता नी आवड़ तेलु हुतु।
Decoded:       खोकडी ने तरता नी आवड़ तेलु हुतु।
Original repr: 'खोकडी ने तरता नी आवड़ तेलु हुतु।'
Decoded repr:  'खोकडी ने तरता नी आवड़ तेलु हुतु।'
Lengths:       31 -> 32
Pieces:        ['▁खोक', 'डी', '▁ने', '

In [25]:

# Section 4.6 — Diagnose Bhili tokenizer round-trip mismatches

import unicodedata
import re

tokenizer_name = "Indic -> Indic 1B"
tok = tokenizer_results[tokenizer_name]

target_col = "Bhili"
df = all_data["bhili"]["validation"].head(100)

def normalize_nfc(text):
    return unicodedata.normalize("NFC", text)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

exact = nfc_only = whitespace_only = both = unresolved = 0

for original in df[target_col].astype(str):
    decoded = tok.tgt_spm.decode(
        tok.tgt_spm.encode(original, out_type=str)
    )

    if decoded == original:
        exact += 1
    elif normalize_nfc(decoded) == normalize_nfc(original):
        nfc_only += 1
    elif normalize_whitespace(decoded) == normalize_whitespace(original):
        whitespace_only += 1
    elif normalize_whitespace(normalize_nfc(decoded)) == normalize_whitespace(
        normalize_nfc(original)
    ):
        both += 1
    else:
        unresolved += 1

print("\nBHILI")
print(f"Examples checked:               {len(df)}")
print(f"Exact matches:                  {exact}")
print(f"NFC resolves mismatch:          {nfc_only}")
print(f"Whitespace resolves mismatch:   {whitespace_only}")
print(f"Both NFC + whitespace resolve:  {both}")
print(f"Still different:                {unresolved}")



BHILI
Examples checked:               100
Exact matches:                  93
NFC resolves mismatch:          7
Whitespace resolves mismatch:   0
Both NFC + whitespace resolve:  0
Still different:                0



# Section 4.6 — Tokenizer Compatibility and Round-Trip Analysis

### 1. Objective

Before beginning model fine-tuning and vocabulary extension, we investigated the pretrained IndicTrans2 tokenizers to understand their vocabulary structure, tokenization behavior on Bhili text, and ability to preserve text through SentencePiece encoding and decoding.

The translation task examined in this notebook is Hindi → Bhili. The investigation focuses on the Indic → Indic 1B checkpoint as an initial candidate for the task.

### 2. Tokenizer and Vocabulary Findings

Five candidate IndicTrans2 checkpoints were inspected. Their tokenizers expose separate source and target SentencePiece models.

| Candidate checkpoint | Source vocabulary | Target vocabulary |
|---|---:|---:|
| Indic → English 1B | 128k | 32k |
| English → Indic 1B | 32k | 128k |
| Indic → Indic 1B | 128k | 128k |
| English → Indic distilled 200M | 32k | 128k |
| Indic → Indic distilled 320M | 128k | 128k |

Fingerprinting the SentencePiece vocabularies identified two distinct vocabulary types: a 32k English-side vocabulary and a 128k Indic-side vocabulary.

The inspected tokenizer special-token IDs were consistent: BOS = 0, PAD = 1, EOS = 2, and UNK = 3.

### 3. Bhili Tokenization Fertility Analysis

The fertility experiment compared the segmentation of Bhili target text using the English-side and Indic-side SentencePiece vocabularies.

| Metric | English-side 32k vocabulary | Indic-side 128k vocabulary |
|---|---:|---:|
| Average tokens per sentence | 106.305 | 30.902 |
| Tokens per whitespace-separated word | 4.980 | 1.448 |
| Word split rate | 98.360% | 31.921% |
| Unknown-piece rate | 0% | 0% |

The Indic-side vocabulary produced substantially fewer tokens per sentence and per word than the English-side vocabulary on the sampled Bhili text.

The zero unknown-piece rate indicates that the sampled text could be segmented without producing unknown pieces under either tested vocabulary. It does not establish that the pretrained model understands or can accurately translate Bhili.

These findings suggest that the Indic-side vocabulary is a more efficient representation of the inspected Bhili text. However, tokenization efficiency alone does not establish translation quality or prove that vocabulary extension will improve performance.

### 4. Language Support and Preprocessing

The installed `IndicTransToolkit` package was searched for references to Bhili, Gondi, Mundari, Santali, and common language codes. The search found `eng_latn` and `hin_deva` in `evaluator.py`, but no explicit references to Bhili in the searched files.

The tokenizer inspection examined language-related attributes, source and target SentencePiece models, preprocessing attributes, and special tokens. These checks did not establish explicit Bhili support.

The checkpoint configuration inspection was also intended to identify language metadata, including language codes and language-to-ID mappings. The available diagnostic results did not conclusively establish whether the selected checkpoint supports Bhili as a target language.

This distinction is important because tokenizer representation, acceptance of a language code, and actual translation capability are separate properties. The ability to tokenize Bhili text does not prove that the pretrained model was trained to generate Bhili.

Further verification of language-tag handling and the complete inference pipeline is therefore required before finalizing the baseline training procedure.

### 5. SentencePiece Round-Trip Experiment

A round-trip experiment was performed on 100 Bhili validation targets using the target SentencePiece model of the Indic → Indic 1B checkpoint.

Each target sentence was encoded into SentencePiece pieces and decoded back into text. The decoded string was then compared with the original using exact string equality.

| Metric | Result |
|---|---:|
| Validation examples checked | 100 |
| Exact matches | 93/100 |
| Exact-match rate | 93% |
| Initial mismatches | 7 |

The first experiment showed that 93 of the 100 Bhili sentences were reconstructed exactly, while seven differed at the string level.

### 6. Investigation of Round-Trip Mismatches

The seven mismatches were investigated using Unicode NFC normalization and whitespace normalization. Each example was assigned to the first category that resolved its mismatch.

| Category | Bhili examples |
|---|---:|
| Exact matches before normalization | 93 |
| Mismatches resolved by NFC normalization | 7 |
| Mismatches resolved by whitespace normalization alone | 0 |
| Mismatches resolved by both normalizations | 0 |
| Still different after the tested checks | 0 |

All seven initial mismatches were resolved by NFC Unicode normalization. None required whitespace normalization, and no differences remained after the tested normalization checks.

These findings suggest that the observed mismatches arose from canonically equivalent Unicode representations rather than unrecoverable text changes. The result is specific to the 100 validation examples examined.

The original dataset was not modified during this diagnostic. NFC normalization was used only to determine whether the mismatches could be explained by Unicode representation differences.

### 7. Interpretation and Implications

The tokenizer experiments provide three useful findings for the Bhili research task.

1. **Vocabulary structure:** The five candidate checkpoints use two distinct SentencePiece vocabulary types, with the Indic-facing vocabulary containing 128k pieces and the English-facing vocabulary containing 32k pieces.

2. **Tokenization efficiency:** The Indic-side vocabulary represents the sampled Bhili text more compactly, with 1.448 tokens per whitespace-separated word compared with 4.980 for the English-side vocabulary.

3. **Text reconstruction:** Although seven of the 100 Bhili examples initially failed exact string comparison, all seven mismatches were resolved by NFC normalization.

For subsequent experiments, the original train, validation, and test data should be preserved. Any normalization policy adopted for training should be documented and applied consistently across the relevant splits.

The proposed vocabulary-extension experiment should preserve existing token IDs and their pretrained embedding rows while ensuring that the tokenizer, embedding matrix, output projection, and vocabulary configuration remain consistent.

The benefits of vocabulary extension must ultimately be measured through held-out translation quality and relevant tokenization metrics, rather than round-trip equality alone.

### 8. Provisional Conclusion

The Section 4.6 investigation establishes that the Indic-side SentencePiece vocabulary segments the sampled Bhili text more compactly than the English-side vocabulary. It also establishes that the selected Indic → Indic 1B tokenizer can reconstruct all 100 sampled Bhili targets after NFC normalization.

However, these results are preliminary tokenizer diagnostics. They do not establish official Bhili support, successful Hindi-to-Bhili inference, or improvements from vocabulary extension.

The next step is to verify the selected checkpoint's language-tag requirements and inference pipeline, establish a working baseline, and then compare baseline LoRA fine-tuning against vocabulary extension with informed embedding initialization using the same held-out evaluation data and procedure.


In [26]:

# Section 4.7 — Inspect language-code preprocessing behavior

from IndicTransToolkit import IndicProcessor

ip = IndicProcessor(inference=True)

source_example = ["This is a short translation test."]
hindi_example = ["यह एक छोटा अनुवाद परीक्षण है।"]

tests = [
    ("Known English -> Santali", source_example, "eng_Latn", "sat_Olck"),
    ("Unknown target-code probe", source_example, "eng_Latn", "xyz_Test"),
    ("Hindi -> Bhili probe", hindi_example, "hin_Deva", "bhili_Deva"),
]

for label, examples, src_lang, tgt_lang in tests:
    print(f"\n{'=' * 60}")
    print(label)
    print(f"Source code: {src_lang} | Target code: {tgt_lang}")

    try:
        result = ip.preprocess_batch(
            examples,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
        )
        print("Preprocessing accepted.")
        print("Output:", result)
    except Exception as e:
        print("Preprocessing rejected.")
        print("Error type:", type(e).__name__)
        print("Error:", str(e)[:500])



Known English -> Santali
Source code: eng_Latn | Target code: sat_Olck
Preprocessing accepted.
Output: ['eng_Latn sat_Olck This is a short translation test .']

Unknown target-code probe
Source code: eng_Latn | Target code: xyz_Test
Preprocessing accepted.
Output: ['eng_Latn xyz_Test This is a short translation test .']

Hindi -> Bhili probe
Source code: hin_Deva | Target code: bhili_Deva
Preprocessing accepted.
Output: ['hin_Deva bhili_Deva यह एक छोटा अनुवाद परीक्षण है ।']


## 5. Language Conditioning and Model Compatibility


In [27]:

# Section 5.2: Inspect language-conditioning interfaces
# Diagnostic only; no model training or dataset modification.

import inspect
from transformers import AutoConfig

MODEL_TO_INSPECT = "ai4bharat/indictrans2-indic-indic-1B"

print("Model:", MODEL_TO_INSPECT)

# Inspect the configuration without loading model weights.
config = AutoConfig.from_pretrained(
    MODEL_TO_INSPECT,
    trust_remote_code=True,
    token=hf_token
)

print("\n--- Model configuration ---")
for key in [
    "model_type",
    "architectures",
    "vocab_size",
    "decoder_start_token_id",
    "bos_token_id",
    "eos_token_id",
    "pad_token_id",
    "tie_word_embeddings",
]:
    print(f"{key}: {getattr(config, key, None)}")

# Reuse the tokenizer already loaded in the notebook.
tok = tokenizer_results["Indic -> Indic 1B"]

print("\n--- Tokenizer class and special tokens ---")
print("Tokenizer class:", type(tok).__name__)
print("Tokenizer length:", len(tok))
print("Configuration vocabulary size:", getattr(config, "vocab_size", None))
print("Special tokens map:", getattr(tok, "special_tokens_map", None))
print("Added vocabulary size:", len(tok.get_added_vocab()))

# Inspect language-related tokenizer attributes.
print("\n--- Tokenizer attributes related to language ---")
for attr in dir(tok):
    if any(term in attr.lower() for term in [
        "lang", "src", "tgt", "prefix", "special", "spm"
    ]):
        try:
            value = getattr(tok, attr)
            if not callable(value):
                print(f"{attr}: {str(value)[:300]}")
        except Exception as exc:
            print(f"{attr}: <could not inspect: {type(exc).__name__}>")

print("\n--- Tokenizer call signature ---")
try:
    print(inspect.signature(tok.__call__))
except Exception as exc:
    print("Could not inspect:", repr(exc))

print("\n--- Tokenizer class methods mentioning language or prefix ---")
for name in dir(type(tok)):
    if any(term in name.lower() for term in [
        "lang", "prefix", "prepare", "encode"
    ]):
        method = getattr(type(tok), name, None)
        if callable(method):
            try:
                print(f"{name}{inspect.signature(method)}")
            except Exception:
                print(name)


Model: ai4bharat/indictrans2-indic-indic-1B

--- Model configuration ---
model_type: IndicTrans
architectures: ['IndicTransForConditionalGeneration']
vocab_size: 122672
decoder_start_token_id: 2
bos_token_id: 0
eos_token_id: 2
pad_token_id: 1
tie_word_embeddings: True

--- Tokenizer class and special tokens ---
Tokenizer class: IndicTransTokenizer
Tokenizer length: 122706
Configuration vocabulary size: 122672
Special tokens map: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}
Added vocabulary size: 4

--- Tokenizer attributes related to language ---
SPECIAL_TOKENS_ATTRIBUTES: ['bos_token', 'eos_token', 'unk_token', 'sep_token', 'pad_token', 'cls_token', 'mask_token', 'additional_special_tokens']
_additional_special_tokens: []
additional_special_tokens: []
additional_special_tokens_ids: []
all_special_ids: [0, 2, 3, 1]
all_special_tokens: ['<s>', '</s>', '<unk>', '<pad>']
all_special_tokens_extended: [AddedToken("<s>", rstrip=False, lstrip=False, si

In [28]:

# Section 5.3: Inspect the language-tag extension method
# Diagnostic only — no vocabulary or model weights are modified.

import inspect

tok = tokenizer_results["Indic -> Indic 1B"]
tok_class = type(tok)

print("Tokenizer class:", tok_class)
print("\n--- add_new_language_tags source ---")

try:
    print(inspect.getsource(tok_class.add_new_language_tags))
except (AttributeError, OSError, TypeError) as exc:
    print("Source unavailable:", repr(exc))

print("\n--- Relevant tokenizer attributes ---")
for attr in [
    "src_vocab_size",
    "tgt_vocab_size",
    "src_encoder",
    "tgt_encoder",
    "src_decoder",
    "tgt_decoder",
    "src_spm",
    "tgt_spm",
]:
    value = getattr(tok, attr, None)

    if isinstance(value, dict):
        print(f"{attr}: dictionary with {len(value)} entries")
    elif value is not None:
        print(f"{attr}: {type(value).__name__} = {value}")
    else:
        print(f"{attr}: not available")

print("\n--- Language-tag examples already in the vocabularies ---")

src_encoder = getattr(tok, "src_encoder", {})
tgt_encoder = getattr(tok, "tgt_encoder", {})

for tag in [
    "eng_Latn",
    "hin_Deva",
    "sat_Olck",
    "bhil",
    "gon",
    "unrecognized_Test",
]:
    src_id = src_encoder.get(tag) if isinstance(src_encoder, dict) else None
    tgt_id = tgt_encoder.get(tag) if isinstance(tgt_encoder, dict) else None

    print(f"{tag!r}: source_id={src_id}, target_id={tgt_id}")


Tokenizer class: <class 'transformers_modules.ai4bharat.indictrans2-indic-indic-1B.24d732922a0a91d0998d5568e3af37b7a21cd705.tokenization_indictrans.IndicTransTokenizer'>

--- add_new_language_tags source ---
    def add_new_language_tags(self, new_tags: List[str]) -> None:
        global LANGUAGE_TAGS
        LANGUAGE_TAGS = frozenset(LANGUAGE_TAGS | set(new_tags))


--- Relevant tokenizer attributes ---
src_vocab_size: int = 122706
tgt_vocab_size: int = 122672
src_encoder: dictionary with 122706 entries
tgt_encoder: dictionary with 122672 entries
src_decoder: dictionary with 122706 entries
tgt_decoder: dictionary with 122672 entries
src_spm: SentencePieceProcessor = <sentencepiece.SentencePieceProcessor object at 0x7e0ee88c1d30>
tgt_spm: SentencePieceProcessor = <sentencepiece.SentencePieceProcessor object at 0x7e0ee8830550>

--- Language-tag examples already in the vocabularies ---
'eng_Latn': source_id=4, target_id=None
'hin_Deva': source_id=8, target_id=None
'sat_Olck': source_id=2

In [29]:

# Section 5.6: Inspect the Bhili pilot input format
# This only preprocesses a sample sentence; it does not train a model.

from IndicTransToolkit import IndicProcessor

pilot_processor = IndicProcessor(inference=True)

cfg = {
    "source_lang": "hin_Deva",
    "target_lang": "mar_Deva",  # Marathi proxy; not an official Bhili tag
    "data_key": "bhili",
    "source_column": "Hindi",
    "target_column": "Bhili",
}

print(f"\n{'=' * 20} Bhili {'=' * 20}")

row = all_data[cfg["data_key"]]["train"].iloc[0]
source_text = str(row[cfg["source_column"]])
target_text = str(row[cfg["target_column"]])

processed_source = pilot_processor.preprocess_batch(
    [source_text],
    src_lang=cfg["source_lang"],
    tgt_lang=cfg["target_lang"],
)

print("Source language tag:", cfg["source_lang"])
print("Target/proxy tag:", cfg["target_lang"])
print("Original source:", repr(source_text))
print("Reference Bhili target:", repr(target_text))
print("Processed source:", repr(processed_source[0]))



==================== Bhili ====================
Source language tag: hin_Deva
Target/proxy tag: mar_Deva
Original source: 'इंफ्रास्ट्रक्चर और टेक्नोलॉजी निवेश में वृद्धि हो रही है।'
Reference Bhili target: 'इन्\u200dफ्रास्\u200dट्रक्\u200dचर मानकों नी रीतें एआई अनुसंधान अनें विकास करवा नों प्रोत्\u200dसाहन मळिरों से।'
Processed source: 'hin_Deva mar_Deva इंफ्रास्ट्रक्चर और टेक्नोलॉजी निवेश में वृद्धि हो रही है ।'


In [30]:

# Section 5.7: Inspect source prefixes and target labels
# Diagnostic only — no model training or tokenizer modification.

from transformers import AutoTokenizer
from IndicTransToolkit import IndicProcessor

MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"

check_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=hf_token
)

pilot_processor = IndicProcessor(inference=True)

# Bhili pilot configuration
source_lang = "hin_Deva"
target_lang = "mar_Deva"  # Marathi proxy, not a verified Bhili tag

row = all_data["bhili"]["train"].iloc[0]

source_text = str(row["Hindi"])
target_text = str(row["Bhili"])

processed_source = pilot_processor.preprocess_batch(
    [source_text],
    src_lang=source_lang,
    tgt_lang=target_lang,
)[0]

# Encode the processed Hindi source and raw Bhili reference independently.
source_ids = check_tokenizer(
    processed_source,
    add_special_tokens=True
)["input_ids"]

target_ids = check_tokenizer(
    text_target=target_text,
    add_special_tokens=True
)["input_ids"]

print("Source language tag:", source_lang)
print("Target/proxy tag:", target_lang)

print("\nSource prefix:", processed_source.split()[:2])
print("Source first token IDs:", source_ids[:12])
print(
    "Source first tokens:",
    check_tokenizer.convert_ids_to_tokens(source_ids[:12])
)

print("\nTarget original text:", repr(target_text[:160]))
print("Target first token IDs:", target_ids[:12])
print(
    "Target first tokens:",
    check_tokenizer.convert_ids_to_tokens(target_ids[:12])
)

print("\nEOS token ID:", check_tokenizer.eos_token_id)
print("Source ends with EOS:", source_ids[-1] == check_tokenizer.eos_token_id)
print("Target ends with EOS:", target_ids[-1] == check_tokenizer.eos_token_id)


Source language tag: hin_Deva
Target/proxy tag: mar_Deva

Source prefix: ['hin_Deva', 'mar_Deva']
Source first token IDs: [8, 18, 2301, 24303, 26, 17663, 4837, 19, 2905, 66, 177, 11]
Source first tokens: ['hin_Deva', 'mar_Deva', '▁इं', 'फ्रास्ट्रक्चर', '▁और', '▁टेक्नोलॉजी', '▁निवेश', '▁में', '▁वृद्धि', '▁हो', '▁रही', '▁है']

Target original text: 'इन्\u200dफ्रास्\u200dट्रक्\u200dचर मानकों नी रीतें एआई अनुसंधान अनें विकास करवा नों प्रोत्\u200dसाहन मळिरों से।'
Target first token IDs: [1905, 106389, 8362, 2525, 106389, 3549, 112, 106389, 4130, 1044, 2130, 385]
Target first tokens: ['▁तुम्ही', 'ੴ', '▁परबर्ती', '▁वाय', 'ੴ', 'ग्रस्', '▁اور', 'ੴ', '्यास', '▁कॊ', 'सार', '▁इत्']

EOS token ID: 2
Source ends with EOS: True
Target ends with EOS: True


In [31]:

# Section 5.8: Compare Bhili target-side tokenization across checkpoints
# Diagnostic only — does not train or modify model weights.

from transformers import AutoTokenizer

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

# Use an actual Bhili training example.
sample_text = str(all_data["bhili"]["train"].iloc[0]["Bhili"])

for model_label, model_id in TOKENIZER_MODELS.items():
    print(f"\n{'=' * 18} {model_label} {'=' * 18}")

    current_tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token,
    )

    print("Tokenizer:", type(current_tok).__name__)

    ids = current_tok(
        text_target=sample_text,
        add_special_tokens=True,
    )["input_ids"]

    pieces = current_tok.convert_ids_to_tokens(ids[:20])

    decoded = current_tok.decode(
        ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    print("\nLanguage: Bhili")
    print("Original:", repr(sample_text[:140]))
    print("Token pieces:", pieces)
    print("Decoded:", repr(decoded[:180]))
    print("Exact round trip:", decoded == sample_text)
    print("Token count (including special tokens):", len(ids))



================== Indic -> Indic 1B ==================
Tokenizer: IndicTransTokenizer

Language: Bhili
Original: 'इन्\u200dफ्रास्\u200dट्रक्\u200dचर मानकों नी रीतें एआई अनुसंधान अनें विकास करवा नों प्रोत्\u200dसाहन मळिरों से।'
Token pieces: ['▁तुम्ही', 'ੴ', '▁परबर्ती', '▁वाय', 'ੴ', 'ग्रस्', '▁اور', 'ੴ', '्यास', '▁कॊ', 'सार', '▁इत्', '▁बाड़ि', '▁अयिते', '▁उम्म', 'िवु', '▁राहुल', '▁अयिते', 'गळिगॆ', 'दे']
Decoded: 'इन्\u200dफ्रास्\u200dट्रक्\u200dचर मानकों नी रीतें एआई अनुसंधान अनें विकास करवा नों प्रोत्\u200dसाहन मळिरों से।'
Exact round trip: True
Token count (including special tokens): 32

================== English -> Indic 1B ==================
Tokenizer: IndicTransTokenizer

Language: Bhili
Original: 'इन्\u200dफ्रास्\u200dट्रक्\u200dचर मानकों नी रीतें एआई अनुसंधान अनें विकास करवा नों प्रोत्\u200dसाहन मळिरों से।'
Token pieces: ['▁Vijay', '<unk>', '▁Hebrew', '▁worldwide', '<unk>', '▁Global', '▁up', '<unk>', '▁booked', '▁alone', '▁receiving', '▁L', '▁Long', '▁received', '▁greet', '▁18

In [32]:

# Section 5.9: Measure Bhili target tokenization quality
# No model weights are loaded or changed.

from transformers import AutoTokenizer
import pandas as pd

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

TARGET_COLUMN = "Bhili"
LANGUAGE = "bhili"
NUM_EXAMPLES = 100

texts = (
    all_data[LANGUAGE]["validation"][TARGET_COLUMN]
    .dropna()
    .astype(str)
    .head(NUM_EXAMPLES)
    .tolist()
)

if not texts:
    raise ValueError("No Bhili validation examples were found.")

results = []

for model_label, model_id in TOKENIZER_MODELS.items():
    print(f"\nLoading {model_label}...")

    current_tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token,
    )

    unk_id = current_tok.unk_token_id

    unk_count = 0
    total_tokens = 0
    affected_sentences = 0
    roundtrip_exact = 0

    for text in texts:
        ids = current_tok(
            text_target=text,
            add_special_tokens=False,
        )["input_ids"]

        sentence_unk_count = (
            sum(token_id == unk_id for token_id in ids)
            if unk_id is not None
            else 0
        )

        unk_count += sentence_unk_count
        total_tokens += len(ids)

        if sentence_unk_count:
            affected_sentences += 1

        decoded = current_tok.decode(
            ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        if decoded == text:
            roundtrip_exact += 1

    result = {
        "model": model_label,
        "language": LANGUAGE,
        "examples": len(texts),
        "unknown_tokens": unk_count,
        "total_tokens": total_tokens,
        "unknown_token_rate_pct": (
            100 * unk_count / total_tokens
            if total_tokens and unk_id is not None
            else 0.0
        ),
        "sentences_with_unknown_pct": (
            100 * affected_sentences / len(texts)
        ),
        "exact_roundtrip_pct": (
            100 * roundtrip_exact / len(texts)
        ),
    }

    results.append(result)

    print(
        f"{LANGUAGE:8} | "
        f"UNK={unk_count:6} | "
        f"tokens={total_tokens:7} | "
        f"UNK rate={result['unknown_token_rate_pct']:7.3f}% | "
        f"sentences affected={result['sentences_with_unknown_pct']:6.1f}% | "
        f"exact round-trip={result['exact_roundtrip_pct']:6.1f}%"
    )

tokenization_report = pd.DataFrame(results)
display(tokenization_report)



Loading Indic -> Indic 1B...
bhili    | UNK=     2 | tokens=   3260 | UNK rate=  0.061% | sentences affected=   1.0% | exact round-trip=  92.0%

Loading English -> Indic 1B...
bhili    | UNK=     2 | tokens=   3260 | UNK rate=  0.061% | sentences affected=   1.0% | exact round-trip=  92.0%


,model,language,examples,unknown_tokens,total_tokens,unknown_token_rate_pct,sentences_with_unknown_pct,exact_roundtrip_pct
0,Indic -> Indic 1B,bhili,100,2,3260,0.06135,1.0,92.0
1,English -> Indic 1B,bhili,100,2,3260,0.06135,1.0,92.0


In [33]:

# Section 5.9.1: Verify whether the two tokenizers produce identical IDs
# Diagnostic only — no model weights are loaded or changed.

from transformers import AutoTokenizer

MODEL_IDS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

texts = (
    all_data["bhili"]["validation"]["Bhili"]
    .dropna()
    .astype(str)
    .head(100)
    .tolist()
)

tokenizers = {}

for label, model_id in MODEL_IDS.items():
    tokenizers[label] = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token,
    )

tok_a = tokenizers["Indic -> Indic 1B"]
tok_b = tokenizers["English -> Indic 1B"]

print("Tokenizer classes:")
print(type(tok_a))
print(type(tok_b))

print("\nVocabulary sizes:")
print("Indic -> Indic:", len(tok_a))
print("English -> Indic:", len(tok_b))

same_ids = 0
same_decoded = 0

for text in texts:
    ids_a = tok_a(text_target=text, add_special_tokens=False)["input_ids"]
    ids_b = tok_b(text_target=text, add_special_tokens=False)["input_ids"]

    if ids_a == ids_b:
        same_ids += 1

    decoded_a = tok_a.decode(
        ids_a, skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )
    decoded_b = tok_b.decode(
        ids_b, skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    if decoded_a == decoded_b:
        same_decoded += 1

print(f"\nExamples with identical token IDs: {same_ids}/{len(texts)}")
print(f"Examples with identical decoded text: {same_decoded}/{len(texts)}")
print(
    "Same vocabulary size:",
    len(tok_a) == len(tok_b)
)


Tokenizer classes:
<class 'transformers_modules.ai4bharat.indictrans2-indic-indic-1B.24d732922a0a91d0998d5568e3af37b7a21cd705.tokenization_indictrans.IndicTransTokenizer'>
<class 'transformers_modules.ai4bharat.indictrans2-en-indic-1B.89cd0007fd7b9e0693a2fd4404d187e07dac79fd.tokenization_indictrans.IndicTransTokenizer'>

Vocabulary sizes:
Indic -> Indic: 122706
English -> Indic: 32322

Examples with identical token IDs: 100/100
Examples with identical decoded text: 100/100
Same vocabulary size: False


In [34]:

# Section 5.9.2: Compare underlying SentencePiece vocabularies
# Diagnostic only — no model weights are loaded or changed.

def compare_spm_vocab(tok_a, tok_b, attr):
    sp_a = getattr(tok_a, attr, None)
    sp_b = getattr(tok_b, attr, None)

    print(f"\n--- {attr} ---")

    if sp_a is None or sp_b is None:
        print("One or both SentencePiece processors are unavailable.")
        return

    size_a = sp_a.get_piece_size()
    size_b = sp_b.get_piece_size()

    print("Vocabulary sizes:", size_a, "vs", size_b)
    print("Same vocabulary size:", size_a == size_b)

    if size_a == size_b:
        same_order = all(
            sp_a.id_to_piece(i) == sp_b.id_to_piece(i)
            for i in range(size_a)
        )
        print("Identical pieces in identical ID order:", same_order)
    else:
        shared_prefix = 0
        for i in range(min(size_a, size_b)):
            if sp_a.id_to_piece(i) != sp_b.id_to_piece(i):
                break
            shared_prefix += 1

        print("Matching initial pieces:", shared_prefix)

tok_a = tokenizers["Indic -> Indic 1B"]
tok_b = tokenizers["English -> Indic 1B"]

compare_spm_vocab(tok_a, tok_b, "src_spm")
compare_spm_vocab(tok_a, tok_b, "tgt_spm")



--- src_spm ---
Vocabulary sizes: 128000 vs 32000
Same vocabulary size: False
Matching initial pieces: 5

--- tgt_spm ---
Vocabulary sizes: 128000 vs 128000
Same vocabulary size: True
Identical pieces in identical ID order: True


## Excellent — this explains why both checkpoints produced identical Bhili tokenization results. The important finding is that their target-side SentencePiece vocabularies are identical, while their source-side vocabularies differ substantially.

In [35]:

# Inspect tokenizer configuration and one Bhili target example
# Diagnostic only — no model weights are trained or modified.

from transformers import AutoTokenizer

TOKENIZER_MODELS = {
    "Indic -> Indic 1B": "ai4bharat/indictrans2-indic-indic-1B",
    "English -> Indic 1B": "ai4bharat/indictrans2-en-indic-1B",
}

example = str(
    all_data["bhili"]["validation"]["Bhili"].dropna().iloc[0]
)

for label, model_id in TOKENIZER_MODELS.items():
    print(f"\n{'=' * 65}")
    print(label)

    tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=hf_token,
    )

    print("Tokenizer class:", type(tok).__name__)
    print("Tokenizer length:", len(tok))
    print("UNK token ID:", tok.unk_token_id)

    for attr in [
        "src_vocab_size",
        "tgt_vocab_size",
        "src_encoder",
        "tgt_encoder",
        "src_spm",
        "tgt_spm",
    ]:
        obj = getattr(tok, attr, None)

        if obj is None:
            print(f"{attr}: unavailable")
        elif isinstance(obj, dict):
            print(f"{attr}: {len(obj)} entries")
        elif isinstance(obj, (int, str)):
            print(f"{attr}: {obj}")
        else:
            print(f"{attr}: {type(obj).__name__}")

    print("\nBhili target tokenization example:")
    encoded = tok(
        text_target=example,
        add_special_tokens=False,
    )
    ids = encoded["input_ids"]

    print("Target text:", repr(example[:150]))
    print("Token count:", len(ids))
    print("First 20 IDs:", ids[:20])
    print("First 20 token strings:", tok.convert_ids_to_tokens(ids[:20]))
    print("UNK IDs in target encoding:", sum(
        i == tok.unk_token_id for i in ids
    ))



Indic -> Indic 1B
Tokenizer class: IndicTransTokenizer
Tokenizer length: 122706
UNK token ID: 3
src_vocab_size: 122706
tgt_vocab_size: 122672
src_encoder: 122706 entries
tgt_encoder: 122672 entries
src_spm: SentencePieceProcessor
tgt_spm: SentencePieceProcessor

Bhili target tokenization example:
Target text: 'बिजा ब्रोह शब्\u200dदों मा समानता एक व्\u200dयक्ति ना वचमाना से की आपडी ताकत अनुसार खर्चा करे।'
Token count: 25
First 20 IDs: [90644, 336, 28692, 25701, 106389, 11131, 354, 16677, 24, 38108, 106389, 254, 7923, 46, 4608, 17831, 20, 12, 149, 516]
First 20 token strings: ['▁मोनबा', 'ऩ्', '▁मैसेज', '▁नयं', 'ੴ', 'वागबहुदु', 'ऱ', 'तिआ', '▁छे', '▁इंश्योरेंस', 'ੴ', 'ब', '▁प्रतिष्ठित', 'urd_Arab', '▁a', '▁सैतं', '▁?', '▁के', '▁करा', 'ध']
UNK IDs in target encoding: 0

English -> Indic 1B
Tokenizer class: IndicTransTokenizer
Tokenizer length: 32322
UNK token ID: 3
src_vocab_size: 32322
tgt_vocab_size: 122672
src_encoder: 32322 entries
tgt_encoder: 122672 entries
src_spm: SentencePieceProc

In [36]:

# Section 5.10: Inspect model architecture and embedding dimensions
# Diagnostic only — no model weights are modified.

import torch
from transformers import AutoConfig, AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

print("Model architecture:", config.architectures)
print("Model type:", config.model_type)
print("Configured vocabulary size:", config.vocab_size)
print("Tied word embeddings:", config.tie_word_embeddings)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

def print_embedding_shape(label, module):
    if module is None:
        print(f"{label}: unavailable")
        return

    weight = getattr(module, "weight", None)
    print(
        f"{label}:",
        tuple(weight.shape) if weight is not None else "no weight matrix"
    )

print("\n--- Input and output embeddings ---")

print_embedding_shape(
    "Input embeddings",
    model.get_input_embeddings()
)

print_embedding_shape(
    "Output embeddings",
    model.get_output_embeddings()
)

for component_name in ("encoder", "decoder"):
    component = getattr(model, component_name, None)

    if component is None and hasattr(model, "model"):
        component = getattr(model.model, component_name, None)

    print(f"\n{component_name.capitalize()} component:")

    if component is not None:
        print_embedding_shape(
            f"{component_name} token embeddings",
            getattr(component, "embed_tokens", None)
        )
    else:
        print("Component not found at the expected attribute path.")

print("\n--- Memory cleanup ---")
del model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Model deleted; CUDA cache cleared if available.")


Model architecture: ['IndicTransForConditionalGeneration']
Model type: IndicTrans
Configured vocabulary size: 122672
Tied word embeddings: True


modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]


--- Input and output embeddings ---
Input embeddings: (122706, 1024)
Output embeddings: (122672, 1024)

Encoder component:
encoder token embeddings: (122706, 1024)

Decoder component:
decoder token embeddings: (122672, 1024)

--- Memory cleanup ---
Model deleted; CUDA cache cleared if available.


In [37]:

import inspect
import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

decoder_weight = model.get_decoder().embed_tokens.weight
output_weight = model.get_output_embeddings().weight

print("=== Parameter comparison ===")
print("Decoder shape:", tuple(decoder_weight.shape))
print("Output shape:", tuple(output_weight.shape))
print("Same object:", decoder_weight is output_weight)
print(
    "Same underlying storage:",
    decoder_weight.untyped_storage().data_ptr()
    == output_weight.untyped_storage().data_ptr()
)
print(
    "Exactly equal values:",
    torch.equal(decoder_weight, output_weight)
)

print("\n=== Model implementation ===")

for method_name in ("tie_weights", "get_output_embeddings"):
    method = getattr(type(model), method_name, None)
    print(f"\n{method_name}:")

    if method is None:
        print("Method not found on the model class.")
        continue

    try:
        print(inspect.getsource(method))
    except (OSError, TypeError) as exc:
        print("Source unavailable:", repr(exc))

print("\n=== Model class ===")
print(type(model))
print("Model source file:", inspect.getfile(type(model)))

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


=== Parameter comparison ===
Decoder shape: (122672, 1024)
Output shape: (122672, 1024)
Same object: False
Same underlying storage: False
Exactly equal values: False

=== Model implementation ===

tie_weights:
    def tie_weights(self):
        if self.config.share_decoder_input_output_embed:
            self._tie_or_clone_weights(self.model.decoder.embed_tokens, self.lm_head)


get_output_embeddings:
    def get_output_embeddings(self):
        return self.lm_head


=== Model class ===
<class 'transformers_modules.ai4bharat.indictrans2-indic-indic-1B.24d732922a0a91d0998d5568e3af37b7a21cd705.modeling_indictrans.IndicTransForConditionalGeneration'>
Model source file: /root/.cache/huggingface/modules/transformers_modules/ai4bharat/indictrans2-indic-indic-1B/24d732922a0a91d0998d5568e3af37b7a21cd705/modeling_indictrans.py


In [38]:
from transformers import AutoConfig

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

print("tie_word_embeddings:",
      getattr(config, "tie_word_embeddings", None))

print("share_decoder_input_output_embed:",
      getattr(config, "share_decoder_input_output_embed", None))

print("\nAll config attributes related to embedding/tied weights:")

for key, value in config.to_dict().items():
    key_lower = key.lower()

    if (
        "embed" in key_lower
        or "tie" in key_lower
        or "share" in key_lower
        or "vocab" in key_lower
    ):
        print(f"{key}: {value}")

tie_word_embeddings: True
share_decoder_input_output_embed: False

All config attributes related to embedding/tied weights:
encoder_vocab_size: 122706
decoder_vocab_size: 122672
layernorm_embedding: False
encoder_embed_dim: 1024
decoder_embed_dim: 1024
scale_embedding: True
share_decoder_input_output_embed: False
tie_word_embeddings: True
tie_encoder_decoder: False
vocab_size: 122672


In [39]:
from transformers import AutoTokenizer

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

print("=== Vocabulary dimensions ===")
print("Tokenizer length:", len(tok))
print("Source vocabulary size:", tok.src_vocab_size)
print("Target vocabulary size:", tok.tgt_vocab_size)

print("\n=== Token ID range checks ===")

for name, vocab in [
    ("Source", tok.src_encoder),
    ("Target", tok.tgt_encoder),
]:
    ids = list(vocab.values())

    print(f"\n{name}:")
    print("Number of entries:", len(vocab))
    print("Minimum token ID:", min(ids))
    print("Maximum token ID:", max(ids))
    print("Unique token IDs:", len(set(ids)))
    print("Contiguous IDs:", set(ids) == set(range(max(ids) + 1)))

print("\n=== Special and language token IDs ===")

for token in [
    "<s>", "<pad>", "</s>", "<unk>",
    "eng_Latn", "hin_Deva", "mar_Deva", "sat_Olck",
]:
    print(
        repr(token),
        "| source ID:", tok.src_encoder.get(token),
        "| target ID:", tok.tgt_encoder.get(token),
    )

=== Vocabulary dimensions ===
Tokenizer length: 122706
Source vocabulary size: 122706
Target vocabulary size: 122672

=== Token ID range checks ===

Source:
Number of entries: 122706
Minimum token ID: 0
Maximum token ID: 122705
Unique token IDs: 122706
Contiguous IDs: True

Target:
Number of entries: 122672
Minimum token ID: 0
Maximum token ID: 122671
Unique token IDs: 122672
Contiguous IDs: True

=== Special and language token IDs ===
'<s>' | source ID: 0 | target ID: 0
'<pad>' | source ID: 1 | target ID: 1
'</s>' | source ID: 2 | target ID: 2
'<unk>' | source ID: 3 | target ID: 3
'eng_Latn' | source ID: 4 | target ID: None
'hin_Deva' | source ID: 8 | target ID: None
'mar_Deva' | source ID: 18 | target ID: None
'sat_Olck' | source ID: 29925 | target ID: None


# 6. Training Pipeline Validation


In [40]:

# Section 5.11: One-example training smoke test for Hindi -> Bhili
# Diagnostic only — no optimizer step or model weights updated.

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IndicTransToolkit import IndicProcessor

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
)

processor = IndicProcessor(inference=True)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

model.train()

# Bhili dataset columns
src_col, tgt_col = "Hindi", "Bhili"
row = all_data["bhili"]["train"].iloc[0]

source_text = str(row[src_col])
target_text = str(row[tgt_col])

# Marathi is a proxy target tag for this diagnostic only.
# It is not a verified Bhili language tag.
processed_source = processor.preprocess_batch(
    [source_text],
    src_lang="hin_Deva",
    tgt_lang="mar_Deva",
)[0]

source_batch = tok(
    processed_source,
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

target_batch = tok(
    text_target=target_text,
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

input_ids = source_batch["input_ids"].to(DEVICE)
attention_mask = source_batch["attention_mask"].to(DEVICE)
labels = target_batch["input_ids"].to(DEVICE)

print("Device:", DEVICE)
print("Input shape:", tuple(input_ids.shape))
print("Labels shape:", tuple(labels.shape))
print("Labels min/max:", labels.min().item(), labels.max().item())
print("Decoder vocabulary size:", model.config.decoder_vocab_size)

assert labels.min().item() >= 0
assert labels.max().item() < model.config.decoder_vocab_size

model.zero_grad(set_to_none=True)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    use_cache=False,
)

print("\nForward-pass loss:", outputs.loss.item())
print("Loss finite:", torch.isfinite(outputs.loss).item())
print("Logits shape:", tuple(outputs.logits.shape))

assert torch.isfinite(outputs.loss).item(), "Loss is not finite!"

outputs.loss.backward()

gradient_count = sum(
    parameter.grad is not None
    for parameter in model.parameters()
)

print("Parameters receiving gradients:", gradient_count)
print("Backward pass completed successfully.")

del outputs, model, tok, processor
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Device: cuda
Input shape: (1, 14)
Labels shape: (1, 32)
Labels min/max: 2 106389
Decoder vocabulary size: 122672

Forward-pass loss: 9.40625
Loss finite: True
Logits shape: (1, 32, 122672)
Parameters receiving gradients: 763
Backward pass completed successfully.


In [41]:

# Section 5.12: Inspect attention-related modules
# Diagnostic only — no model weights are modified.

import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

attention_modules = []

for name, module in model.named_modules():
    class_name = module.__class__.__name__.lower()

    if "attention" in class_name or "attn" in class_name:
        attention_modules.append((name, module.__class__.__name__))

print("Attention-related modules:")
for name, class_name in attention_modules:
    print(f"{name}: {class_name}")

print("\nTotal attention-related modules:", len(attention_modules))

print("\nLinear layers inside attention-related modules:")
for module_name, module in model.named_modules():
    class_name = module.__class__.__name__.lower()

    if ("attention" in class_name or "attn" in class_name):
        for child_name, child in module.named_modules():
            if child_name and isinstance(child, torch.nn.Linear):
                print(
                    f"{module_name}.{child_name}: "
                    f"{child.in_features} -> {child.out_features}"
                )

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Attention-related modules:
model.encoder.layers.0.self_attn: IndicTransAttention
model.encoder.layers.1.self_attn: IndicTransAttention
model.encoder.layers.2.self_attn: IndicTransAttention
model.encoder.layers.3.self_attn: IndicTransAttention
model.encoder.layers.4.self_attn: IndicTransAttention
model.encoder.layers.5.self_attn: IndicTransAttention
model.encoder.layers.6.self_attn: IndicTransAttention
model.encoder.layers.7.self_attn: IndicTransAttention
model.encoder.layers.8.self_attn: IndicTransAttention
model.encoder.layers.9.self_attn: IndicTransAttention
model.encoder.layers.10.self_attn: IndicTransAttention
model.encoder.layers.11.self_attn: IndicTransAttention
model.encoder.layers.12.self_attn: IndicTransAttention
model.encoder.layers.13.self_attn: IndicTransAttention
model.encoder.layers.14.self_attn: IndicTransAttention
model.encoder.layers.15.self_attn: IndicTransAttention
model.encoder.layers.16.self_attn: IndicTransAttention
model.encoder.layers.17.self_attn: IndicTransAtt

In [42]:

# Section 5.13: Check training environment versions

import torch
import transformers
import peft
import importlib.metadata as metadata

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

for package in ["torchao", "accelerate", "tokenizers", "huggingface_hub"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")


PyTorch: 2.11.0+cu128
Transformers: 4.46.3
PEFT: 0.20.0
torchao: 0.10.0
accelerate: 1.14.0
tokenizers: 0.20.3
huggingface_hub: 0.36.2


In [43]:

# Section 5.14: Inspect attention projection layers for LoRA
# Diagnostic only — no model weights are modified.

import torch
from transformers import AutoModelForSeq2SeqLM

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

for module_name in [
    "model.encoder.layers.0.self_attn",
    "model.decoder.layers.0.self_attn",
    "model.decoder.layers.0.encoder_attn",
]:
    print(f"\n{module_name}")
    print("-" * len(module_name))

    try:
        module = model.get_submodule(module_name)
    except AttributeError:
        print("Module path not found.")
        continue

    print("Module class:", type(module).__name__)
    found_linear = False

    for name, submodule in module.named_modules():
        if isinstance(submodule, torch.nn.Linear):
            found_linear = True
            print(
                f"{name or '<self>'}: "
                f"{submodule.in_features} -> {submodule.out_features}"
            )

    if not found_linear:
        print("No torch.nn.Linear layers found in this module.")

# Free memory after inspection.
del model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nModel unloaded; inspection complete.")



model.encoder.layers.0.self_attn
--------------------------------
Module class: IndicTransAttention
k_proj: 1024 -> 1024
v_proj: 1024 -> 1024
q_proj: 1024 -> 1024
out_proj: 1024 -> 1024

model.decoder.layers.0.self_attn
--------------------------------
Module class: IndicTransAttention
k_proj: 1024 -> 1024
v_proj: 1024 -> 1024
q_proj: 1024 -> 1024
out_proj: 1024 -> 1024

model.decoder.layers.0.encoder_attn
-----------------------------------
Module class: IndicTransAttention
k_proj: 1024 -> 1024
v_proj: 1024 -> 1024
q_proj: 1024 -> 1024
out_proj: 1024 -> 1024

Model unloaded; inspection complete.


In [44]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [45]:

# Section 5.15: Verify LoRA target modules before applying adapters

import torch
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

# List actual linear-layer names that could be LoRA targets.
linear_names = set()

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        linear_names.add(name.split(".")[-1])

print("Available linear-layer names:")
print(sorted(linear_names))

# Start with the proposed targets, but fail clearly if they are absent.
requested_targets = ["q_proj", "v_proj"]
missing_targets = [
    target for target in requested_targets
    if target not in linear_names
]

if missing_targets:
    print("\nRequested LoRA targets not found:", missing_targets)
    print("Do not apply LoRA yet. Inspect the attention projection names above.")
else:
    LORA_CONFIG = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=requested_targets,
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_2_SEQ_LM",
    )

    lora_model = get_peft_model(model, LORA_CONFIG)
    lora_model.print_trainable_parameters()

    trainable_params = sum(
        p.numel() for p in lora_model.parameters() if p.requires_grad
    )
    total_params = sum(p.numel() for p in lora_model.parameters())

    print(f"\nTrainable parameters: {trainable_params:,}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")

    print("\nLoRA-wrapped modules:")
    for name, module in lora_model.named_modules():
        if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
            print(name)


Available linear-layer names:
['fc1', 'fc2', 'k_proj', 'lm_head', 'out_proj', 'q_proj', 'v_proj']
trainable params: 3,538,944 || all params: 1,211,635,712 || trainable%: 0.2921

Trainable parameters: 3,538,944
Total parameters: 1,211,635,712
Trainable percentage: 0.2921%

LoRA-wrapped modules:
base_model.model.model.encoder.layers.0.self_attn.v_proj
base_model.model.model.encoder.layers.0.self_attn.q_proj
base_model.model.model.encoder.layers.1.self_attn.v_proj
base_model.model.model.encoder.layers.1.self_attn.q_proj
base_model.model.model.encoder.layers.2.self_attn.v_proj
base_model.model.model.encoder.layers.2.self_attn.q_proj
base_model.model.model.encoder.layers.3.self_attn.v_proj
base_model.model.model.encoder.layers.3.self_attn.q_proj
base_model.model.model.encoder.layers.4.self_attn.v_proj
base_model.model.model.encoder.layers.4.self_attn.q_proj
base_model.model.model.encoder.layers.5.self_attn.v_proj
base_model.model.model.encoder.layers.5.self_attn.q_proj
base_model.model.mode

In [46]:
# Verify trainable parameter names and confirm the base model is frozen.

trainable_names = [
    name
    for name, param in lora_model.named_parameters()
    if param.requires_grad
]

print("Number of trainable parameter tensors:", len(trainable_names))
print("\nFirst 20 trainable parameter names:")

for name in trainable_names[:20]:
    print(name)

unexpected_trainable = [
    name for name in trainable_names
    if "lora_A" not in name and "lora_B" not in name
]

print("\nUnexpected trainable parameters:", len(unexpected_trainable))

assert len(trainable_names) > 0
assert len(unexpected_trainable) == 0, (
    "Some trainable parameters are not LoRA A/B adapters."
)

print("\nPASS: Only LoRA adapter parameters are trainable.")

Number of trainable parameter tensors: 216

First 20 trainable parameter names:
base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.encoder.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.encoder.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.encoder.layers.1.self_attn.v_proj.lora_A.default.weight
base_model.model.model.encoder.layers.1.self_attn.v_proj.lora_B.default.weight
base_model.model.model.encoder.layers.1.self_attn.q_proj.lora_A.default.weight
base_model.model.model.encoder.layers.1.self_attn.q_proj.lora_B.default.weight
base_model.model.model.encoder.layers.2.self_attn.v_proj.lora_A.default.weight
base_model.model.model.encoder.layers.2.self_attn.v_proj.lora_B.default.weight
base_model.model.model.encoder.layers.2.self_attn.q_proj.lora_A.default.weight
base_model.model.model.encoder.layers.2.self_attn.q

In [47]:

# Prepare Bhili training and validation examples for LoRA fine-tuning.

import pandas as pd

from IndicTransToolkit import IndicProcessor

processor = IndicProcessor(inference=True)

print("IndicProcessor initialized successfully.")

LANGUAGE = "bhili"
SRC_LANG = "hin_Deva"
TGT_LANG = "mar_Deva"  # Proxy tag only; Bhili support is not yet verified.
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

SRC_COL = "Hindi"
TGT_COL = "Bhili"

train_df = all_data[LANGUAGE]["train"].copy()
val_df = all_data[LANGUAGE]["validation"].copy()

def prepare_examples(df):
    examples = []

    for _, row in df.iterrows():
        source_text = str(row[SRC_COL]).strip()
        target_text = str(row[TGT_COL]).strip()

        if not source_text or not target_text:
            continue

        processed_source = processor.preprocess_batch(
            [source_text],
            src_lang=SRC_LANG,
            tgt_lang=TGT_LANG,
        )[0]

        examples.append({
            "source": processed_source,
            "target": target_text,
        })

    return examples

train_examples = prepare_examples(train_df)
val_examples = prepare_examples(val_df)

print("Language:", LANGUAGE)
print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))

if train_examples:
    print("\nPrepared example:")
    print("Source:", train_examples[0]["source"])
    print("Target:", train_examples[0]["target"])

assert len(train_examples) > 0, "No training examples were prepared."
assert len(val_examples) > 0, "No validation examples were prepared."


IndicProcessor initialized successfully.
Language: bhili
Training examples: 17817
Validation examples: 990

Prepared example:
Source: hin_Deva mar_Deva इंफ्रास्ट्रक्चर और टेक्नोलॉजी निवेश में वृद्धि हो रही है ।
Target: इन्‍फ्रास्‍ट्रक्‍चर मानकों नी रीतें एआई अनुसंधान अनें विकास करवा नों प्रोत्‍साहन मळिरों से।


In [48]:

from transformers import AutoTokenizer

MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"

tok = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=hf_token,
)

print("Tokenizer loaded:", type(tok).__name__)
print("Tokenizer length:", len(tok))
print("Encoder vocabulary size:", model.config.encoder_vocab_size)
print("Decoder vocabulary size:", model.config.decoder_vocab_size)


import torch
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 4

class TranslationDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_translation_batch(examples):
    sources = [example["source"] for example in examples]
    targets = [example["target"] for example in examples]

    source_batch = tok(
        sources,
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt",
    )

    target_batch = tok(
        text_target=targets,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt",
    )

    labels = target_batch["input_ids"].clone()
    labels[target_batch["attention_mask"] == 0] = -100

    return {
        "input_ids": source_batch["input_ids"],
        "attention_mask": source_batch["attention_mask"],
        "labels": labels,
    }


train_dataset = TranslationDataset(train_examples)
val_dataset = TranslationDataset(val_examples)

assert len(train_dataset) > 0, "Training dataset is empty."
assert len(val_dataset) > 0, "Validation dataset is empty."

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_translation_batch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_translation_batch,
)

batch = next(iter(train_loader))

for key, value in batch.items():
    print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")

print("\nLabels contain -100:", (batch["labels"] == -100).any().item())

source_max_id = batch["input_ids"].max().item()
valid_labels = batch["labels"][batch["labels"] != -100]

print(
    "Input IDs within encoder vocabulary:",
    source_max_id < model.config.encoder_vocab_size,
)

print(
    "Target IDs within decoder vocabulary:",
    valid_labels.numel() > 0
    and valid_labels.max().item() < model.config.decoder_vocab_size,
)

assert source_max_id < model.config.encoder_vocab_size
assert valid_labels.numel() > 0
assert valid_labels.max().item() < model.config.decoder_vocab_size

print("\nPASS: Batch construction and vocabulary checks succeeded.")


Tokenizer loaded: IndicTransTokenizer
Tokenizer length: 122706
Encoder vocabulary size: 122706
Decoder vocabulary size: 122672
input_ids: shape=(4, 55), dtype=torch.int64
attention_mask: shape=(4, 55), dtype=torch.int64
labels: shape=(4, 68), dtype=torch.int64

Labels contain -100: True
Input IDs within encoder vocabulary: True
Target IDs within decoder vocabulary: True

PASS: Batch construction and vocabulary checks succeeded.


In [49]:
from torch.optim import AdamW

# Keep the first test small; this is not the full training loop.
LEARNING_RATE = 1e-4

optimizer = AdamW(
    (p for p in lora_model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
)

lora_model.train()
lora_model.zero_grad(set_to_none=True)

batch = {
    key: value.to(DEVICE)
    for key, value in batch.items()
}

# Snapshot one trainable parameter before the update.
tracked_name, tracked_param = next(
    (name, param)
    for name, param in lora_model.named_parameters()
    if param.requires_grad and "lora_B" in name
)
parameter_before = tracked_param.detach().clone()

outputs = lora_model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"],
    use_cache=False,
)

loss = outputs.loss
print("Loss before update:", loss.item())
assert torch.isfinite(loss).item(), "Loss is not finite."

loss.backward()


tracked_grad = dict(lora_model.named_parameters())[tracked_name].grad

assert tracked_grad is not None, "Tracked LoRA parameter has no gradient."
assert torch.isfinite(tracked_grad).all().item(), (
    "Tracked LoRA parameter has non-finite gradients."
)

print("Tracked LoRA gradient norm:", tracked_grad.norm().item())


grad_norm = torch.nn.utils.clip_grad_norm_(
    [p for p in lora_model.parameters() if p.requires_grad],
    max_norm=1.0,
)
print("Gradient norm:", grad_norm.item())
assert torch.isfinite(grad_norm).item(), "Gradient norm is not finite."

optimizer.step()

parameter_after = dict(lora_model.named_parameters())[tracked_name].detach()
changed = not torch.equal(parameter_before, parameter_after)

print("Tracked parameter:", tracked_name)
print("Parameter changed after optimizer step:", changed)

assert changed, "The tracked LoRA parameter did not change."
print("\nPASS: One optimizer step completed successfully.")

Loss before update: 8.9609375
Tracked LoRA gradient norm: 0.0091173080727458
Gradient norm: 0.9603366851806641
Tracked parameter: base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight
Parameter changed after optimizer step: True

PASS: One optimizer step completed successfully.


# 7. Baseline LoRA Fine-Tuning


In [50]:

# Bhili baseline LoRA training configuration

import random
import numpy as np
import torch

SEED = 42
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 10                 # Maximum epochs
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0

EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 0.001

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Experiment: Bhili baseline — original vocabulary + LoRA")
print("Maximum epochs:", NUM_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Minimum validation-loss improvement:", EARLY_STOPPING_MIN_DELTA)
print("Micro-batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Warmup ratio:", WARMUP_RATIO)
print("Device:", DEVICE)


Experiment: Bhili baseline — original vocabulary + LoRA
Maximum epochs: 10
Early stopping patience: 2
Minimum validation-loss improvement: 0.001
Micro-batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Learning rate: 0.0001
Warmup ratio: 0.03
Device: cuda


In [51]:

import gc
import torch
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"

# Release previous model instances and GPU memory.
if "lora_model" in globals():
    del lora_model
if "model" in globals():
    del model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load a fresh copy of the original pretrained checkpoint.
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=hf_token,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

# Create fresh LoRA adapters.
LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

lora_model = get_peft_model(model, LORA_CONFIG)

# Verify trainable parameters.
lora_model.print_trainable_parameters()

trainable_names = [
    name for name, param in lora_model.named_parameters()
    if param.requires_grad
]

unexpected_trainable = [
    name for name in trainable_names
    if "lora_A" not in name and "lora_B" not in name
]

assert trainable_names, "No trainable LoRA parameters found."
assert not unexpected_trainable, (
    f"Unexpected trainable parameters: {unexpected_trainable[:10]}"
)

print("\nModel:", MODEL_ID)
print("Device:", DEVICE)
print("Maximum epochs:", NUM_EPOCHS)
print("Early-stopping patience:", EARLY_STOPPING_PATIENCE)
print("Original vocabulary retained:", True)
print("Fresh LoRA adapters initialized:", True)
print("Only LoRA parameters trainable:", True)


trainable params: 3,538,944 || all params: 1,211,635,712 || trainable%: 0.2921

Model: ai4bharat/indictrans2-indic-indic-1B
Device: cuda
Maximum epochs: 10
Early-stopping patience: 2
Original vocabulary retained: True
Fresh LoRA adapters initialized: True
Only LoRA parameters trainable: True


In [52]:

import math
from pathlib import Path
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Reproducible Bhili baseline output directory
BASELINE_DIR = Path("/kaggle/working/bhili_baseline_lora")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

# Number of optimizer updates in one epoch
updates_per_epoch = math.ceil(
    len(train_loader) / GRADIENT_ACCUMULATION_STEPS
)
total_training_steps = updates_per_epoch * NUM_EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

optimizer = AdamW(
    (p for p in lora_model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print("Training batches per epoch:", len(train_loader))
print("Optimizer updates per epoch:", updates_per_epoch)
print("Maximum optimizer updates:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("Initial learning rate:", optimizer.param_groups[0]["lr"])
print("Checkpoint directory:", BASELINE_DIR)

assert len(train_loader) > 0
assert len(val_loader) > 0
assert total_training_steps > 0

print("\nPASS: Optimizer and scheduler are ready.")


Training batches per epoch: 4455
Optimizer updates per epoch: 1114
Maximum optimizer updates: 11140
Warmup steps: 334
Initial learning rate: 0.0
Checkpoint directory: /kaggle/working/bhili_baseline_lora

PASS: Optimizer and scheduler are ready.


## 7.3 Baseline Training Loop


In [53]:

import gc
import json
import math
import random
import time
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import torch
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Correct output directory and safety checks
# --------------------------------------------------
BASELINE_DIR = Path("/kaggle/working/bhili_baseline_lora")
BEST_DIR = BASELINE_DIR / "best_adapter"
LATEST_DIR = BASELINE_DIR / "latest_adapter"
RECOVERY_DIR = BASELINE_DIR / "recovery"

for directory in [BASELINE_DIR, BEST_DIR, LATEST_DIR, RECOVERY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

assert LANGUAGE == "bhili", f"Wrong language: {LANGUAGE}"
assert len(train_loader) > 0 and len(val_loader) > 0
assert lora_model is not None
assert GRADIENT_ACCUMULATION_STEPS >= 1
assert NUM_EPOCHS >= 1

print("Output directory:", BASELINE_DIR)
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Training batches per epoch:", len(train_loader))
print("Validation batches per epoch:", len(val_loader))
print("Maximum epochs:", NUM_EPOCHS)
print("Device:", DEVICE)
print("Started:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Existing output files will not be automatically deleted.", flush=True)

# Avoid accidentally treating old history as part of this run.
history_path = BASELINE_DIR / "training_history.json"
if history_path.exists():
    raise FileExistsError(
        f"{history_path} already exists. Inspect or back it up before "
        "starting a fresh run; do not overwrite earlier results."
    )

# --------------------------------------------------
# 2. Training state
# --------------------------------------------------
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
history = []
global_step = 0
completed_batches = 0

updates_per_epoch = math.ceil(
    len(train_loader) / GRADIENT_ACCUMULATION_STEPS
)
planned_updates = updates_per_epoch * NUM_EPOCHS

training_start = time.time()
optimizer.zero_grad(set_to_none=True)

# --------------------------------------------------
# 3. Training and validation
# --------------------------------------------------
for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    lora_model.train()
    train_losses = []
    num_batches = len(train_loader)

    train_bar = tqdm(
        enumerate(train_loader, start=1),
        total=num_batches,
        desc=f"Epoch {epoch}/{NUM_EPOCHS} — Training",
        unit="batch",
        dynamic_ncols=True,
    )

    for batch_idx, batch in train_bar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = lora_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
            use_cache=False,
        )

        loss = outputs.loss
        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite training loss: epoch {epoch}, batch {batch_idx}"
            )

        train_losses.append(loss.item())

        # Scale by the actual number of micro-batches in this group.
        group_start = (
            (batch_idx - 1) // GRADIENT_ACCUMULATION_STEPS
        ) * GRADIENT_ACCUMULATION_STEPS
        group_size = min(
            GRADIENT_ACCUMULATION_STEPS,
            num_batches - group_start,
        )
        (loss / group_size).backward()

        should_update = (
            batch_idx % GRADIENT_ACCUMULATION_STEPS == 0
            or batch_idx == num_batches
        )

        if should_update:
            grad_norm = torch.nn.utils.clip_grad_norm_(
                [p for p in lora_model.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )
            if not torch.isfinite(grad_norm).item():
                raise RuntimeError("Non-finite gradient norm.")

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        completed_batches += 1
        elapsed = time.time() - training_start

        # ETA is approximate and includes training batches only.
        avg_batch_time = elapsed / completed_batches
        remaining_batches = max(
            0, planned_updates * 0 + len(train_loader) * NUM_EPOCHS
            - completed_batches
        )
        eta_seconds = avg_batch_time * remaining_batches

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{np.mean(train_losses):.4f}",
            updates=f"{global_step}/{planned_updates}",
            elapsed=str(timedelta(seconds=int(elapsed))),
            ETA=str(timedelta(seconds=int(eta_seconds))),
        )

    train_bar.close()

    # Validation
    lora_model.eval()
    val_losses = []

    val_bar = tqdm(
        val_loader,
        total=len(val_loader),
        desc=f"Epoch {epoch}/{NUM_EPOCHS} — Validation",
        unit="batch",
        dynamic_ncols=True,
    )

    with torch.no_grad():
        for batch in val_bar:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            outputs = lora_model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
                use_cache=False,
            )

            val_loss = outputs.loss
            if not torch.isfinite(val_loss).item():
                raise RuntimeError(
                    f"Non-finite validation loss at epoch {epoch}"
                )

            val_losses.append(val_loss.item())
            val_bar.set_postfix(
                val_loss=f"{np.mean(val_losses):.4f}"
            )

    val_bar.close()

    if not val_losses:
        raise RuntimeError("Validation loader produced no batches.")

    mean_train_loss = float(np.mean(train_losses))
    mean_val_loss = float(np.mean(val_losses))
    epoch_seconds = time.time() - epoch_start

    record = {
        "epoch": epoch,
        "train_loss": mean_train_loss,
        "validation_loss": mean_val_loss,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "optimizer_steps": global_step,
        "epoch_seconds": epoch_seconds,
        "timestamp": datetime.now().isoformat(),
    }
    history.append(record)

    print(
        f"\nEpoch {epoch}/{NUM_EPOCHS} completed\n"
        f"Train loss: {mean_train_loss:.4f} | "
        f"Validation loss: {mean_val_loss:.4f} | "
        f"Duration: {epoch_seconds / 60:.1f} min",
        flush=True,
    )

    # Save latest adapter after every completed epoch.
    lora_model.save_pretrained(LATEST_DIR)

    # Update best adapter when validation loss improves sufficiently.
    if mean_val_loss < best_val_loss - EARLY_STOPPING_MIN_DELTA:
        best_val_loss = mean_val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        lora_model.save_pretrained(BEST_DIR)
        print(f"Saved best adapter: epoch {epoch}", flush=True)
    else:
        epochs_without_improvement += 1
        print(
            f"No sufficient improvement: "
            f"{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}",
            flush=True,
        )

    # Persist history and metadata every epoch.
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    metadata = {
        "experiment": "Bhili baseline — original vocabulary + LoRA",
        "model_id": MODEL_ID,
        "language": LANGUAGE,
        "source_column": SRC_COL,
        "target_column": TGT_COL,
        "source_language_tag": SRC_LANG,
        "target_language_tag": TGT_LANG,
        "best_epoch": best_epoch,
        "best_validation_loss": best_val_loss,
        "completed_epoch": epoch,
        "optimizer_steps": global_step,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_source_length": MAX_SOURCE_LENGTH,
        "max_target_length": MAX_TARGET_LENGTH,
        "lora_r": LORA_CONFIG.r,
        "lora_alpha": LORA_CONFIG.lora_alpha,
        "lora_dropout": LORA_CONFIG.lora_dropout,
    }

    with open(BASELINE_DIR / "experiment_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    # Save optimizer/scheduler and RNG state for recovery.
    # The latest adapter is saved separately in LATEST_DIR.
    recovery_state = {
        "completed_epoch": epoch,
        "global_step": global_step,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available() else None
        ),
    }

    recovery_tmp = RECOVERY_DIR / "training_state.tmp"
    recovery_path = RECOVERY_DIR / "training_state.pt"
    torch.save(recovery_state, recovery_tmp)
    recovery_tmp.replace(recovery_path)

    print("Saved epoch history, latest adapter, and recovery state.", flush=True)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.", flush=True)
        break

if best_epoch == 0:
    raise RuntimeError(
        "No best adapter was saved. Inspect the training logs and output files."
    )

total_elapsed = time.time() - training_start

summary = {
    "best_epoch": best_epoch,
    "best_validation_loss": best_val_loss,
    "optimizer_steps_completed": global_step,
    "total_training_seconds": total_elapsed,
    "finished_at": datetime.now().isoformat(),
}

with open(BASELINE_DIR / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nTraining finished:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Optimizer updates:", global_step)
print("Total time:", str(timedelta(seconds=int(total_elapsed))))
print("Best adapter:", BEST_DIR)
print("Latest adapter:", LATEST_DIR)
print("Recovery state:", RECOVERY_DIR / "training_state.pt")


Output directory: /kaggle/working/bhili_baseline_lora
Training examples: 17817
Validation examples: 990
Training batches per epoch: 4455
Validation batches per epoch: 248
Maximum epochs: 10
Device: cuda
Started: 2026-10-24 06:59:23
Existing output files will not be automatically deleted.


Epoch 1/10 — Training:   0%|          | 0/4455 [00:00<?, ?batch/s]

Epoch 1/10 — Validation:   0%|          | 0/248 [00:00<?, ?batch/s]


Epoch 1/10 completed
Train loss: 5.8702 | Validation loss: 5.0366 | Duration: 12.8 min
Saved best adapter: epoch 1
Saved epoch history, latest adapter, and recovery state.


Epoch 2/10 — Training:   0%|          | 0/4455 [00:00<?, ?batch/s]

...

Epoch 9/10 — Validation:   0%|          | 0/248 [00:00<?, ?batch/s]


Epoch 9/10 completed
Train loss: 4.5900 | Validation loss: 4.4568 | Duration: 12.7 min
Saved best adapter: epoch 9
Saved epoch history, latest adapter, and recovery state.


Epoch 10/10 — Training:   0%|          | 0/4455 [00:00<?, ?batch/s]

Epoch 10/10 — Validation:   0%|          | 0/248 [00:00<?, ?batch/s]


Epoch 10/10 completed
Train loss: 4.5732 | Validation loss: 4.4630 | Duration: 12.6 min
No sufficient improvement: 1/2
Saved epoch history, latest adapter, and recovery state.

Training finished: 2026-10-24 09:06:15
Best epoch: 9
Best validation loss: 4.456842237903226
Optimizer updates: 11140
Total time: 2:06:52
Best adapter: /kaggle/working/bhili_baseline_lora/best_adapter
Latest adapter: /kaggle/working/bhili_baseline_lora/latest_adapter
Recovery state: /kaggle/working/bhili_baseline_lora/recovery/training_state.pt


In [54]:

from pathlib import Path
import json

BASELINE_DIR = Path("/kaggle/working/bhili_baseline_lora")

assert BASELINE_DIR.exists(), "Baseline output directory is missing."

print("Baseline directory:", BASELINE_DIR)
print("\nSaved files:")

for path in sorted(BASELINE_DIR.rglob("*")):
    if path.is_file():
        print(
            f"{path.relative_to(BASELINE_DIR)} "
            f"({path.stat().st_size / (1024 ** 2):.2f} MB)"
        )

history_path = BASELINE_DIR / "training_history.json"
best_adapter = BASELINE_DIR / "best_adapter"
latest_adapter = BASELINE_DIR / "latest_adapter"
recovery_state = BASELINE_DIR / "recovery" / "training_state.pt"

assert history_path.exists(), "Training history is missing."
assert (best_adapter / "adapter_config.json").exists(), (
    "Best adapter config is missing."
)
assert any(best_adapter.glob("adapter_model.*")), (
    "Best adapter weights are missing."
)
assert latest_adapter.exists(), "Latest adapter directory is missing."
assert recovery_state.exists(), "Recovery state is missing."

history = json.loads(history_path.read_text())

print("\nCompleted epochs:", len(history))
for record in history:
    print(record)

print("\nPASS: History, best adapter, latest adapter, and recovery state exist.")


Baseline directory: /kaggle/working/bhili_baseline_lora

Saved files:
best_adapter/README.md (0.00 MB)
best_adapter/adapter_config.json (0.00 MB)
best_adapter/adapter_model.safetensors (13.53 MB)
experiment_metadata.json (0.00 MB)
latest_adapter/README.md (0.00 MB)
latest_adapter/adapter_config.json (0.00 MB)
latest_adapter/adapter_model.safetensors (13.53 MB)
recovery/training_state.pt (27.19 MB)
training_history.json (0.00 MB)
training_summary.json (0.00 MB)

Completed epochs: 10
{'epoch': 1, 'train_loss': 5.870246299803592, 'validation_loss': 5.036636844758065, 'learning_rate': 9.278178789561354e-05, 'optimizer_steps': 1114, 'epoch_seconds': 768.2742714881897, 'timestamp': '2026-10-24T07:12:11.517582'}
{'epoch': 2, 'train_loss': 5.1002555941358025, 'validation_loss': 4.783092867943548, 'learning_rate': 8.247270035165649e-05, 'optimizer_steps': 2228, 'epoch_seconds': 759.1071772575378, 'timestamp': '2026-10-24T07:24:51.112726'}
{'epoch': 3, 'train_loss': 4.922537878787879, 'validatio

In [55]:

# Evaluate the trained baseline on the held-out test set using BLEU and chrF

import sacrebleu
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm

RESULTS_DIR = Path("/kaggle/working/bhili_baseline_lora/evaluation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load the best trained LoRA adapter
BEST_DIR = "/kaggle/working/bhili_baseline_lora/best_adapter"

if "best_eval" not in lora_model.peft_config:
    lora_model.load_adapter(BEST_DIR, adapter_name="best_eval")
lora_model.set_adapter("best_eval")
lora_model.eval()

# Prepare test examples
test_df = all_data["bhili"]["test"].dropna(
    subset=["Hindi", "Bhili"]
).copy()

sources = test_df["Hindi"].astype(str).str.strip().tolist()
references = test_df["Bhili"].astype(str).str.strip().tolist()

valid = [
    i for i, (src, ref) in enumerate(zip(sources, references))
    if src and ref
]
sources = [sources[i] for i in valid]
references = [references[i] for i in valid]

predictions = []

for start in tqdm(range(0, len(sources), 8), desc="Test evaluation"):
    batch = sources[start:start + 8]

    processed = processor.preprocess_batch(
        batch,
        src_lang=SRC_LANG,
        tgt_lang=TGT_LANG,
    )

    inputs = tok(
        processed,
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.inference_mode():
        output_ids = lora_model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=5,
        )

    decoded = tok.batch_decode(
        output_ids, skip_special_tokens=True
    )
    decoded = processor.postprocess_batch(decoded, lang=TGT_LANG)
    predictions.extend(decoded)

# Calculate corpus-level metrics
bleu = sacrebleu.corpus_bleu(predictions, [references])
chrf = sacrebleu.corpus_chrf(predictions, [references])

print(f"Test examples: {len(references)}")
print(f"BLEU: {bleu.score:.4f}")
print(f"chrF: {chrf.score:.4f}")

# Save metrics and predictions
pd.DataFrame({
    "Hindi": sources,
    "Reference_Bhili": references,
    "Predicted_Bhili": predictions,
}).to_csv(RESULTS_DIR / "baseline_test_predictions.csv", index=False)

with open(RESULTS_DIR / "baseline_test_metrics.txt", "w") as f:
    f.write(f"BLEU: {bleu.score:.4f}\n")
    f.write(f"chrF: {chrf.score:.4f}\n")
    f.write(f"Test examples: {len(references)}\n")

print(f"Saved evaluation results to: {RESULTS_DIR}")


Test evaluation:   0%|          | 0/124 [00:00<?, ?it/s]

Test examples: 990
BLEU: 12.1669
chrF: 43.3445
Saved evaluation results to: /kaggle/working/bhili_baseline_lora/evaluation


In [56]:

# Package baseline model, checkpoints, training logs, and evaluation results

from pathlib import Path
import zipfile
import os

BASELINE_DIR = Path("/kaggle/working/bhili_baseline_lora")
ZIP_PATH = Path("/kaggle/working/bhili_baseline_artifacts.zip")

assert BASELINE_DIR.exists(), (
    f"Baseline folder not found: {BASELINE_DIR}. "
    "Check your training output path."
)

# Check that the best adapter was saved
BEST_DIR = BASELINE_DIR / "best_adapter"
assert BEST_DIR.exists(), (
    f"Best adapter not found: {BEST_DIR}"
)

# Create ZIP (outside the folder being zipped)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for file in BASELINE_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, file.relative_to(BASELINE_DIR.parent))

# Verify the archive
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    contents = zf.namelist()
    bad_file = zf.testzip()

print("ZIP created:", ZIP_PATH)
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"Files packaged: {len(contents)}")
print("Archive integrity:", "PASS" if bad_file is None else f"FAILED: {bad_file}")

print("\nPackaged files:")
for name in contents:
    print(name)

print("\nEvaluation results present:")
for file in BASELINE_DIR.rglob("*test_metrics*"):
    print(file)
for file in BASELINE_DIR.rglob("*test_predictions*"):
    print(file)


ZIP created: /kaggle/working/bhili_baseline_artifacts.zip
ZIP size: 50.07 MB
Files packaged: 12
Archive integrity: PASS

Packaged files:
bhili_baseline_lora/training_history.json
bhili_baseline_lora/experiment_metadata.json
bhili_baseline_lora/training_summary.json
bhili_baseline_lora/latest_adapter/adapter_model.safetensors
bhili_baseline_lora/latest_adapter/README.md
bhili_baseline_lora/latest_adapter/adapter_config.json
bhili_baseline_lora/evaluation/baseline_test_predictions.csv
bhili_baseline_lora/evaluation/baseline_test_metrics.txt
bhili_baseline_lora/recovery/training_state.pt
bhili_baseline_lora/best_adapter/adapter_model.safetensors
bhili_baseline_lora/best_adapter/README.md
bhili_baseline_lora/best_adapter/adapter_config.json

Evaluation results present:
/kaggle/working/bhili_baseline_lora/evaluation/baseline_test_metrics.txt
/kaggle/working/bhili_baseline_lora/evaluation/baseline_test_predictions.csv
